<div style="display: flex; align-items: center;">
    <h1>Tutorial on hybrid crop modelling with diffWOFOST</h1>
    <img src="https://raw.githubusercontent.com/WUR-AI/diffWOFOST/refs/heads/main/docs/logo/diffwofost.png" width="150" style="margin-left: 20px;">
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WUR-AI/diffWOFOST/blob/main/docs/notebooks/hybrid_stress_correction.ipynb)

In this tutorial we will build a **hybrid crop model** by
plugging a small neural network into WOFOST72 and training
it end-to-end. The notebook uses the
[diffwofost](https://github.com/WUR-AI/diffWOFOST) Python package and a public
field-trial dataset.

The network classes (`StressNN`, `NNStressFactor`) live next to this notebook
in [`hybrid_stress.py`](hybrid_stress.py). They are tutorial-specific and are
not part of the installed package.

By the end you will have:

- explored a real field-trial dataset (potato, two Dutch sites, 168 plot-years);
- replaced WOFOST's standard evapotranspiration block with a learnable stress
  factor (`RFTRA`) produced by a neural network;
- compared the hybrid against a potential-production WOFOST72_PP reference *and*
  against a pure-ML LSTM with no physics in the loop;
- seen a teaser of what else differentiability buys you (parameter sensitivities
  for free).

> **Citations.** The field data is from Ten Den et al. (2024), Harvard
> Dataverse [doi:10.7910/DVN/1LC6W7](https://doi.org/10.7910/DVN/1LC6W7),
> CC BY-NC-SA 4.0. The diffwofost engine and this tutorial are
> EUPL-1.1. Pretrained models in this notebook are derivative works of
> the field data and are therefore also released under CC BY-NC-SA 4.0.
> See [DATA_LICENSE.md](DATA_LICENSE.md).


## 1. The big picture: why hybrid modelling?

Mechanistic crop models like **WOFOST** encode decades of agronomic knowledge as
ODEs (Ordinary Differential Equation): photosynthesis, partitioning, water balance, phenology. They are
*interpretable*, *physically grounded*, and *generalise* to weather and
management regimes outside their calibration data — but they have to make
simplifying assumptions, and the assumptions sometimes break.

Pure **data-driven** models (an LSTM that maps weather + soil → yield) can
absorb whatever structure is in the data, but they have **no inductive bias**:
nothing tells them carbon is conserved, that leaves grow before tubers, or that
photosynthesis caps at light saturation. They overfit small datasets badly and
extrapolate poorly outside the training distribution.

**Hybrid models** keep the physics where it's well-understood and let a neural
network fill in the parts that are poorly modelled. Concretely, this tutorial:

| Component | Source |
|-----------|--------|
| Phenology (DVS, TSUM) | physical (WOFOST) |
| Carbon partitioning to leaves/stems/tubers | physical (WOFOST) |
| Soil water balance (Potetial Production variant) | physical (WOFOST) |
| **Stress reduction factor (`RFTRA`)** | **learned NN** |

In standard WOFOST, `RFTRA` represents a transpiration reduction factor:
a scalar that reduces gross assimilation under water-limited conditions.
Strictly speaking, it is intended to capture water stress only.

In this tutorial, however, we deliberately use `RFTRA` more broadly as a
generic stress gate on assimilation. The `_PP` (potential production) variant
of WOFOST does not model nutrient stress, and therefore systematically
overpredicts growth on nitrogen-limited plots. Instead of implementing the full
water- and nitrogen-limited production variants, we train a small neural network
to infer an effective daily stress factor directly from weather and treatment
context.

The physics still governs phenology, carbon allocation, and crop growth
dynamics; the neural network only modulates how much carbon is fixed each day.

**Disclaimer** This is admittedly a modelling shortcut — we are “misusing” `RFTRA` beyond its
original physiological interpretation — but it is defensible because RFTRA
acts multiplicatively on gross carbon assimilation. From the model’s perspective, both drought stress and nitrogen stress ultimately reduce canopy assimilation, even if the underlying physiological mechanisms differ.

> ❓ **Discuss with a neighbour**
>
> What other parts of a crop model might
> be good candidates for a hybrid replacement? What are the trade-offs of
> replacing more physics with NN vs. less?
>


## 2. Run me first

Run the setup cells below. They install diffwofost when needed (Colab),
download the field-trial data and pretrained weights, and fetch
`hybrid_stress.py` if this notebook was opened on its own (Colab copies
the `.ipynb` but not sibling files).


In [ ]:
#@title Install diffwofost { display-mode: "form" }
#@markdown On Colab this installs the package. Local checkouts can skip it if you already have an editable install.

import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "diffwofost", "matplotlib", "pandas", "openpyxl",
    ])
else:
    print("Local session: expecting an existing diffwofost install (e.g. pip install -e .).")


In [ ]:
#@title Imports and global settings { display-mode: "form" }
#@markdown Libraries, `float64` on CPU, and local data paths.

import copy
import inspect
import sys
import warnings
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import openpyxl
from openpyxl import Workbook
from pcse.base import ParameterProvider
from pcse.input import (
    ExcelWeatherDataProvider,
    YAMLAgroManagementReader,
    YAMLCropDataProvider,
    WOFOST72SiteDataProvider,
)
from pcse.util import DummySoilDataProvider

from diffwofost.physical_models.config import ComputeConfig, Configuration
from diffwofost.physical_models.crop.wofost72 import Wofost72
from diffwofost.physical_models.engine import Engine
from diffwofost.physical_models.soil.classic_waterbalance import WaterbalancePP

warnings.filterwarnings("ignore", message="To copy construct from a tensor.*")
ComputeConfig.set_device("cpu")
ComputeConfig.set_dtype(torch.float64)

notebook_dir = Path.cwd()
data_dir = notebook_dir / "data"
data_temp_dir = notebook_dir / "data_temp"
field_data_dir = notebook_dir / "field_data"
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))
print(f"data_dir       = {data_dir}")
print(f"data_temp_dir  = {data_temp_dir}")
print(f"field_data_dir = {field_data_dir}")


### 2.1 Download data and crop parameters

Three sources (see [DATA_LICENSE.md](DATA_LICENSE.md)):

1. **Field-trial files** in [`field_data/`](field_data/) from Ten Den et al.
   (2024), Harvard Dataverse
   [doi:10.7910/DVN/1LC6W7](https://doi.org/10.7910/DVN/1LC6W7), CC BY-NC-SA 4.0.
2. **Potato parameters** from
   [`ajwdewit/WOFOST_crop_parameters`](https://github.com/ajwdewit/WOFOST_crop_parameters)
   (Apache-2.0), one variety block per cultivar. Potential production then sits
   above the observations, which is required because `RFTRA` can only *reduce*
   growth.
3. **PCSE config** (`Wofost72_PP.conf`) from `ajwdewit/pcse` (Apache-2.0).
   Per-cultivar agromanagement files are generated inline.
4. **Pretrained weights** in [`pretrained/`](pretrained/) (CC BY-NC-SA 4.0).
   Later cells load them by default; set `FORCE_RETRAIN = True` to train from
   scratch.

All downloads are cached: re-running the cell is a no-op.


In [ ]:
#@title Download data, crop parameters, and notebook helpers { display-mode: "form" }
#@markdown Field-trial data, potato parameters, `hybrid_stress.py`, and pretrained weights. Re-running is a no-op once cached.

for d in [
    data_temp_dir,
    data_temp_dir / "trained_models",
    data_dir / "conf",
    data_dir / "crop_nl",
    data_dir / "agro",
]:
    d.mkdir(parents=True, exist_ok=True)

# Colab copies only the .ipynb. Sibling files are fetched from GitHub.
# Colab's cell JS runs in a sandboxed iframe, so the blob URL (and thus the
# branch name) is usually invisible — inferred ref falls back to main. Until
# this notebook is on main, also try the tutorial branch.
import os
import re
import subprocess
from urllib.parse import unquote
from urllib.request import Request, urlopen

GITHUB_OWNER, GITHUB_REPO = "WUR-AI", "diffWOFOST"
GITHUB_TUTORIAL_BRANCH = "docs/hybrid-stress-tutorial"
NOTEBOOK_BLOB_RE = re.compile(
    r"/github/[^/]+/[^/]+/blob/(.+)/docs/notebooks/[^/?#]+\.ipynb"
)


def _ref_from_url(url):
    if not url:
        return None
    match = NOTEBOOK_BLOB_RE.search(unquote(url).replace("\\", "/"))
    return match.group(1) if match else None


def infer_github_ref():
    for key in ("DIFFWOFOST_GITHUB_REF", "GITHUB_REF_NAME", "GITHUB_HEAD_REF"):
        val = os.environ.get(key, "").strip()
        if val:
            return val.removeprefix("refs/heads/")
    if "google.colab" in sys.modules:
        try:
            from google.colab import output
            hrefs = output.eval_js(
                """
                (() => {
                  const urls = [];
                  for (const loc of [window.top, window.parent, window]) {
                    try { urls.push(loc.location.href); } catch (e) {}
                  }
                  try { urls.push(document.referrer); } catch (e) {}
                  return urls.filter(Boolean).join("\\n");
                })()
                """,
                timeout_sec=15,
            )
            for href in str(hrefs).splitlines():
                ref = _ref_from_url(href.strip())
                if ref:
                    return ref
        except Exception:
            pass
    try:
        ref = subprocess.check_output(
            ["git", "rev-parse", "--abbrev-ref", "HEAD"],
            cwd=notebook_dir, text=True, stderr=subprocess.DEVNULL,
        ).strip()
        if ref and ref != "HEAD":
            return ref
    except Exception:
        pass
    return "main"


def raw_notebook_bases(refs):
    bases = []
    seen = set()
    for ref in refs:
        if not ref or ref in seen:
            continue
        seen.add(ref)
        bases.append(
            f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}"
            f"/refs/heads/{ref}/docs/notebooks"
        )
    return bases


GITHUB_REF = infer_github_ref()
GITHUB_RAW = raw_notebook_bases(
    [GITHUB_REF, GITHUB_TUTORIAL_BRANCH, "main"]
)
print(f"Notebook GitHub ref: {GITHUB_REF}")
print("Download refs:", ", ".join(
    base.split("/refs/heads/")[1].split("/docs/notebooks")[0]
    for base in GITHUB_RAW
))


def _download(url, dest):
    req = Request(url, headers={"User-Agent": "diffwofost-notebook"})
    with urlopen(req, timeout=30) as src, open(dest, "wb") as out:
        out.write(src.read())


def ensure_notebook_file(rel_path):
    dest = notebook_dir / rel_path
    if dest.exists():
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    last_error = None
    for base in GITHUB_RAW:
        url = f"{base}/{rel_path}"
        try:
            print(f"Downloading {rel_path} from {url}")
            _download(url, dest)
            return dest
        except Exception as exc:
            last_error = exc
            if dest.exists():
                dest.unlink()
            print(f"  failed: {exc}")
    raise RuntimeError(f"Could not download {rel_path}") from last_error


ensure_notebook_file("hybrid_stress.py")
for _ckpt in ("stress_nn_year.pt", "pure_lstm_year.pt"):
    ensure_notebook_file(f"pretrained/{_ckpt}")

# 1. Field-trial data — Ten Den et al. (2024), CC BY-NC-SA 4.0.
#    Shipped in field_data/; copy into gitignored data_temp for conversion.
for name in [
    "Plotspecific_processed.csv",
    "Weatherfile_lelystad.xlsx",
    "Weatherfile_vredepeel.xlsx",
]:
    src = ensure_notebook_file(f"field_data/{name}")
    dest = data_temp_dir / name
    if not dest.exists():
        dest.write_bytes(src.read_bytes())

# 2. PCSE config + per-cultivar potato parameters (Apache-2.0).
#    Parameters come from ajwdewit/WOFOST_crop_parameters (wofost72 branch).
#    Potential production must sit above observed yield: RFTRA can only reduce
#    growth from PP (section 8.2). Files go in crop_nl/ rather than crop/.
PCSE_URL = "https://raw.githubusercontent.com/ajwdewit/pcse/master/pcse/conf"
CROP_URL = "https://raw.githubusercontent.com/ajwdewit/WOFOST_crop_parameters/wofost72"
for url, dest in [
    (f"{PCSE_URL}/Wofost72_PP.conf", data_dir / "conf" / "Wofost72_PP.conf"),
    (f"{CROP_URL}/potato.yaml", data_dir / "crop_nl" / "potato.yaml"),
]:
    if not dest.exists():
        print(f"Downloading {dest.name}...")
        urlretrieve(url, dest)

# crops.yaml is just the index telling YAMLCropDataProvider which crop files to
# expect. The upstream one lists all 20 WOFOST crops and the provider then
# insists on finding barley.yaml, maize.yaml and the rest. We only need potato,
# so write a one-crop index instead of downloading 20 files we won't open.
(data_dir / "crop_nl" / "crops.yaml").write_text(
    "available_crops:\n - potato\n"
)

# 3. Per-cultivar agromanagement.
#    The trial labels cultivars C1..C6. Those codes are decoded by the dataset's
#    own codebook (Plotspecific_processed_meta.xlsx on Harvard Dataverse):
#
#        C1 Innovator | C2 Fontane (Lelystad) | C3 Markies
#        C4 Premiere | C5 Fontane (Vredepeel) | C6 Festien
#
#    so C2 and C5 are the same cultivar grown at the two sites — five distinct
#    cultivars, not six. Each gets its own calibrated parameter block, selected
#    via `variety_name` in the agromanagement file below.
CULTIVAR_TO_VARIETY = {
    "C1": "Innovator", "C2": "Fontane", "C3": "Markies",
    "C4": "Premiere", "C5": "Fontane", "C6": "Festien",
}
VARIETIES = sorted(set(CULTIVAR_TO_VARIETY.values()))

# No crop_end_date: under crop_end_type 'maturity' the campaign ends when DVS
# reaches DVSEND, and the date is never read.
AGRO_TEMPLATE = (
    "Version: 1.0\n"
    "AgroManagement:\n"
    "- {year}-04-20:\n"
    "    CropCalendar:\n"
    "        crop_name: 'potato'\n"
    "        variety_name: '{variety}'\n"
    "        crop_start_date: {year}-04-20\n"
    "        crop_start_type: 'sowing'\n"
    "        crop_end_type: 'maturity'\n"
    "        max_duration: 300\n"
    "    TimedEvents:\n"
    "    StateEvents:\n"
)
for _year in (2019, 2020):
    for _variety in VARIETIES:
        _dest = data_dir / "agro" / f"AGMT_{_variety}_{_year}.agro"
        if not _dest.exists():
            _dest.write_text(AGRO_TEMPLATE.format(year=_year, variety=_variety))
print(f"Wrote per-cultivar agromanagement for {len(VARIETIES)} varieties x 2 years")

conf_path = data_dir / "conf" / "Wofost72_PP.conf"
crop_path = data_dir / "crop_nl"

print("\nAll data ready.")


In [ ]:
#@title Convert weather to PCSE format { display-mode: "form" }
# Convert both data_temp weather files into PCSE-compatible format
PCSE_COLS = ["DAY", "IRRAD", "TMIN", "TMAX", "VAP", "WIND", "RAIN", "SNOWDEPTH"]
UNITS = ["date", "kJ/m2/day or hours", "Celsius", "Celsius", "kPa", "m/sec", "mm", "cm"]


def convert_weather_to_pcse_format(src, dst, *, force=False):
    if dst.exists() and not force:
        return dst
    src_wb = openpyxl.load_workbook(src, data_only=True)
    sh = src_wb.active
    country, station, description = sh.cell(2, 2).value, sh.cell(3, 2).value, sh.cell(4, 2).value
    source_text, nodata_val = sh.cell(5, 2).value, sh.cell(6, 2).value
    longitude, latitude, elevation = sh.cell(8, 1).value, sh.cell(8, 2).value, sh.cell(8, 3).value
    angstrom_a, angstrom_b, has_sunshine = sh.cell(8, 4).value, sh.cell(8, 5).value, sh.cell(8, 6).value
    df = pd.read_excel(src, header=9).iloc[1:].reset_index(drop=True)[PCSE_COLS].copy()
    df["DAY"] = pd.to_datetime(df["DAY"])
    for c in PCSE_COLS[1:]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df.loc[df["SNOWDEPTH"] < 0, "SNOWDEPTH"] = nodata_val

    wb = Workbook(); s = wb.active; s.title = "Weather"
    s.cell(1, 1).value = "Site Characteristics"
    s.cell(2, 1).value = "Country"; s.cell(2, 2).value = country
    s.cell(3, 1).value = "Station"; s.cell(3, 2).value = station
    s.cell(4, 1).value = "Description"; s.cell(4, 2).value = description
    s.cell(5, 1).value = "Source"; s.cell(5, 2).value = source_text
    s.cell(6, 1).value = "Contact"
    s.cell(7, 1).value = "Missing values"; s.cell(7, 2).value = nodata_val
    for j, l in enumerate(["Longitude", "Latitude", "Elevation", "AngstromA", "AngstromB", "HasSunshine"], 1):
        s.cell(8, j).value = l
    for j, v in enumerate([longitude, latitude, elevation, angstrom_a, angstrom_b, has_sunshine], 1):
        s.cell(9, j).value = v
    s.cell(10, 1).value = "Observed data"
    for j, c in enumerate(PCSE_COLS, 1):
        s.cell(11, j).value = c
    for j, u in enumerate(UNITS, 1):
        s.cell(12, j).value = u
    for i, row in enumerate(df.itertuples(index=False), 13):
        for j, v in enumerate(row, 1):
            s.cell(i, j).value = v.to_pydatetime() if hasattr(v, "to_pydatetime") else v
    wb.save(dst)
    return dst


weather_paths = {
    "L": convert_weather_to_pcse_format(
        data_temp_dir / "Weatherfile_lelystad.xlsx",
        data_temp_dir / "Weatherfile_lelystad_pcse.xlsx",
    ),
    "V": convert_weather_to_pcse_format(
        data_temp_dir / "Weatherfile_vredepeel.xlsx",
        data_temp_dir / "Weatherfile_vredepeel_pcse.xlsx",
    ),
}
print(f"Weather files ready: {list(weather_paths.values())}")


## 3. Data inspection

The dataset is a multi-year potato field trial run at two Dutch sites:

- **Lelystad (L)** — clay soil, central Netherlands;
- **Vredepeel (V)** — sandy soil, southeast Netherlands.

The experimental design is nested, not factorial — a distinction that matters later:

```
2 years (2019, 2020) × 3 N-levels × 2 W-levels × replicate plots
  Lelystad (clay): C1 Innovator, C2 Fontane, C3 Markies
  Vredepeel (sand): C4 Premiere, C5 Fontane, C6 Festien
= 168 unique plot-years (after dropping the small shaded sub-trial)
```

No cultivar was grown at both sites, so cultivar and site are entangled: any
"cultivar effect" is also a site effect. With one exception. The codebook that
ships with the dataset decodes the trial's anonymous C-codes, and it reveals
that **C2 and C5 are the same cultivar — Fontane**, grown at Lelystad and at
Vredepeel respectively. Fontane is the only cultivar that spans both sites,
so a five-variety one-hot plus a site bit is not collinear (§6). Read any
later grouping by C-code as **cultivar-and-site**.

Each plot has ~7 destructive biomass samples taken over the growing season, plus
continuous weather records. Let's load it up and have a look under the hood.


In [ ]:
#@title Inspect data { display-mode: "form" }
obs_df = pd.read_csv(data_temp_dir / "Plotspecific_processed.csv")
obs_df["Date"] = pd.to_datetime(obs_df["Date"])

# Drop the small Shadow side-experiment (C2 N2 W2 only on Lelystad, ~60 rows)
obs_df = obs_df[obs_df["Shadow"].isna()].copy()

# Keep only rows with at least one biomass/LAI observation
BIO_COLUMNS = ["LeavesDW", "StemDW", "tubersDW", "LAI"]
ROOT_COLUMN = "rootsDW"
obs_df = obs_df[obs_df[BIO_COLUMNS + [ROOT_COLUMN]].notna().any(axis=1)].copy()

# Per-plot index: (Year, Location, Plotnumber)
plot_keys = (
    obs_df[["Year", "Location", "Plotnumber", "Cultivar", "Nitrogen", "Irrigation"]]
    .drop_duplicates()
    .sort_values(["Year", "Location", "Plotnumber"])
    .reset_index(drop=True)
)
PLOT_KEYS = list(plot_keys.itertuples(index=False, name="Plot"))
print(f"Total plot-years (after Shadow drop): {len(PLOT_KEYS)}")
print()
print("Plot-years per (Year, Location, Cultivar):")
print(plot_keys.groupby(["Year", "Location", "Cultivar"]).size().unstack(fill_value=0))
print()
print("Per-variable observation counts:")
for col in BIO_COLUMNS + [ROOT_COLUMN]:
    print(f"  {col:<10} non-null: {obs_df[col].notna().sum()}")


## 4. Explore the data

Before fitting any model, a visual survey helps us sanity-check the dataset and
form intuition about what a model should be able to learn. Three quick views:

1. biomass / LAI trajectories per cultivar,
2. site-level comparison (Lelystad vs. Vredepeel),
3. treatment-effect preview (does N / W leave a signature?).


### 4.1 Biomass and LAI trajectories per cultivar

We expect leaves to peak around flowering, stems to plateau, tubers to rise
after flowering, and LAI to roughly follow the leaf curve. If anything looks
wildly off here, the data — not the model — is the first place to look.


In [ ]:
#@title Plot: biomass and LAI by cultivar { display-mode: "form" }
#@markdown §4.1 — scatter trajectories per cultivar.

CULTIVAR_COLORS = {c: plt.cm.tab10(i) for i, c in
                   enumerate(sorted(obs_df["Cultivar"].dropna().unique()))}
PLOT_VARIABLES = [
    ("LeavesDW", "Leaf dry matter (g / m²)"),
    ("StemDW", "Stem dry matter (g / m²)"),
    ("tubersDW", "Tuber dry matter (g / m²)"),
    ("LAI", "Leaf area index"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=False)
for ax, (col, title) in zip(axes.ravel(), PLOT_VARIABLES):
    for cultivar in sorted(obs_df["Cultivar"].dropna().unique()):
        subset = obs_df[(obs_df["Cultivar"] == cultivar) & obs_df[col].notna()]
        if subset.empty:
            continue
        ax.scatter(subset["Date"], subset[col],
                   s=16, alpha=0.5,
                   color=CULTIVAR_COLORS[cultivar], label=cultivar)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.tick_params(axis="x", rotation=30)

axes[0, 0].legend(title="Cultivar", loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()


### 4.2 Site comparison

Lelystad vs. Vredepeel side by side. If the two sites differ enough that they
look like distinct populations, our model needs to be told which is which (the
site bit in the input features handles this).


In [ ]:
#@title Plot: site comparison { display-mode: "form" }
#@markdown §4.2 — Lelystad vs Vredepeel.

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=False)
loc_colors = {"L": "tab:blue", "V": "tab:orange"}
for ax, (col, title) in zip(axes.ravel(), PLOT_VARIABLES):
    for loc, sub in obs_df[obs_df[col].notna()].groupby("Location"):
        ax.scatter(sub["Date"], sub[col],
                   s=14, alpha=0.5,
                   color=loc_colors.get(str(loc), "gray"), label=str(loc))
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.tick_params(axis="x", rotation=30)
axes[0, 0].legend(title="Location")
plt.tight_layout(); plt.show()


### 4.3 Treatment-effect preview

Do nitrogen level and irrigation treatment leave a visible mark on the
observations? If they do, the hybrid model has something to learn from them —
the NN gets a one-hot encoding of treatment as input.

> ❓ **Discuss with a neighbour**
>
> Look at the two panels below. Can you see the treatment effect?
>


In [ ]:
#@title Plot: treatment-effect preview { display-mode: "form" }
#@markdown §4.3 — nitrogen and irrigation.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_values = sorted(obs_df["Nitrogen"].dropna().unique())
for n in n_values:
    sub = obs_df[(obs_df["Nitrogen"] == n) & obs_df["tubersDW"].notna()]
    axes[0].scatter(sub["Date"], sub["tubersDW"], s=14, alpha=0.5,
                    label=f"N = {n}")
axes[0].set_title("tubersDW vs Date, by Nitrogen level")
axes[0].set_ylabel("tubersDW (g / m²)")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].tick_params(axis="x", rotation=30)

w_values = sorted(obs_df["Irrigation"].dropna().unique())
for w in w_values:
    sub = obs_df[(obs_df["Irrigation"] == w) & obs_df["LAI"].notna()]
    axes[1].scatter(sub["Date"], sub["LAI"], s=14, alpha=0.5,
                    label=f"W = {w}")
axes[1].set_title("LAI vs Date, by Irrigation level")
axes[1].set_ylabel("LAI")
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout(); plt.show()


## 5. The hybrid model

The hybrid we will build looks like this (simplified):

```
                +-------------------+      +-------------------+
   weather ---> | Phenology (DVS)   | ---> | Partitioning      |
   crop YAML    | (WOFOST physics)  |      | (WOFOST physics)  |
                +-------------------+      +-------------------+
                                                    |
                                                    v
                  +------------------+      +---------------------+
   weather ---->  | Stress NN        | -->  | Gross assimilation  |
   DVS, treatment | (learned RFTRA)  |      | (WOFOST × RFTRA)    |
                  +------------------+      +---------------------+
                                                    |
                                                    v
                                            organ biomass per day
                                                    |
                                            compare to observations
                                                    |
                                            loss --> backprop --> NN
```
**How it works.** In standard WOFOST, the daily stress reduction factor `RFTRA` is computed inside the evapotranspiration module. This factor represents how much crop assimilation should be reduced due to limited transpiration under water stress.
Under the `_PP` (potential production) configuration, however, no water stress
is simulated, and the model simply returns `RFTRA = 1.0` (no stress).

In this tutorial, we replace that evapotranspiration stress component with a
small neural-network module called `NNStressFactor`.

Each simulated day, the WOFOST engine calls this module to ask:
> “By how much should today’s gross assimilation be reduced?”


The replacement module:

1. assembles the day’s feature vector
(weather variables, DVS, and plot/treatment context),
2. feeds those inputs through a small multilayer perceptron (StressNN),
3. returns a sigmoid output as `RFTRA ∈ [0, 1]`.


Importantly, this learned factor is a **lumped reduction of carbon gain**, not
a water-stress or nitrogen-stress submodel. Potential production has no
`WaterbalanceFD` and no NPK module, so both limitations are missing and
`RFTRA` is the one remaining knob on `GASS = PGASS × RFTRA`.
Physiologically, `RFTRA` was meant only for transpiration. Using it as an
any-stress gate is a modelling shortcut: it can rank treatments that reduce
canopy carbon fixation, but it cannot separate water from nitrogen, and it
cannot reproduce N-specific dynamics such as early senescence.

Because the entire simulation engine is implemented in PyTorch, every operation
remains differentiable. The seasonal loss (for example, yield prediction error)
therefore stays connected to the neural-network parameters through the complete
computational graph of the crop model.

Calling `loss.backward()` propagates gradients backward through the entire growing season and into the weights of `StressNN` — which is the central idea behind diffWOFOST.


### 5.1 The two plug-in components

`StressNN` is the tiny MLP that maps daily features → `RFTRA ∈ [0, 1]`.
`NNStressFactor` is the WOFOST evapotranspiration swap that calls it on every
simulated day.

Both live in [`hybrid_stress.py`](hybrid_stress.py) next to this notebook —
they are specific to this tutorial and are not imported from the diffwofost
package. The next cell imports them and prints the source so you can see the
hook. Skip straight to §6 if you would rather look at the features first.


In [ ]:
from hybrid_stress import NNStressFactor
from hybrid_stress import StressNN

print(inspect.getsource(StressNN))
print(inspect.getsource(NNStressFactor))


## 6. Building input features for the NN

The NN gets three groups of inputs:

**Per-plot context (constant over the season).**

| Field | Encoding | Dim |
|-------|----------|-----|
| Site | binary (Lelystad=0, Vredepeel=1) | 1 |
| Nitrogen level | numeric (N0=0.0, N1=0.3, N2=1.3) | 1 |
| Irrigation level | binary (W1=0, W2=1) | 1 |
| Variety | one-hot of the five actual cultivars | 5 |

The trial still labels plots C1–C6, but C2 and C5 are both Fontane, so the
network one-hot is over five varieties rather than six trial codes. Collapsing
them onto one Fontane column is what keeps the site bit from being an exact
sum of three cultivar dummies. That is only an encoding choice so site and
variety are not collinear.

**Per-day weather features (computed once per site).**

| Field | Computed from | Dim |
|-------|---------------|-----|
| VPD (vapor pressure deficit) | `SVP(TMAX) − VAP` | 1 |
| TMAX (max temp) | weather file | 1 |
| 7-day rolling rainfall | weather file | 1 |
| IRRAD (radiation) | weather file (÷ 1e4) | 1 |

**Per-day crop-state features (read from the engine kiosk daily).**

| Field | Dim |
|-------|-----|
| DVS, LAI, TAGP | 3 |

**Total NN input dim: 4 (weather) + 3 (crop state) + 8 (plot context) = 15.**


In [ ]:
#@title Feature encodings: plot context { display-mode: "form" }
#@markdown §6 — site, N level, irrigation, and variety one-hot encodings.

SITE_INDEX = {"L": 0, "V": 1}
N_LEVEL_NUMERIC = {"N0": 0.0, "N1": 0.3, "N2": 1.3}
W_LEVEL_INDEX = {"W1": 0, "W2": 1}

# Trial labels stay C1–C6 (how the data and plots are grouped). The NN one-hot
# is over the five actual varieties: C2 and C5 are both Fontane. That is the
# only cultivar grown at both sites, so site and cultivar are not collinear.
CULTIVARS = ["C1", "C2", "C3", "C4", "C5", "C6"]
CULTIVAR_INDEX = {c: i for i, c in enumerate(CULTIVARS)}
VARIETY_INDEX = {v: i for i, v in enumerate(VARIETIES)}


def make_plot_context_tensor(cultivar, nitrogen, irrigation, location):
    site_bit = float(SITE_INDEX[location])
    n_num = float(N_LEVEL_NUMERIC[nitrogen])
    w_bit = float(W_LEVEL_INDEX[irrigation])
    variety_oh = [0.0] * len(VARIETIES)
    variety_oh[VARIETY_INDEX[CULTIVAR_TO_VARIETY[cultivar]]] = 1.0
    return torch.tensor(
        [site_bit, n_num, w_bit] + variety_oh,
        dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device(),
    )


PLOT_CONTEXT_DIM = 3 + len(VARIETIES)
print(f"Varieties (one-hot columns): {VARIETIES}")
print(f"Plot context dim: {PLOT_CONTEXT_DIM}")
print(f"Example (C2, N2, W2, Lelystad): {make_plot_context_tensor('C2', 'N2', 'W2', 'L')}")
print(f"Example (C5, N2, W2, Vredepeel): {make_plot_context_tensor('C5', 'N2', 'W2', 'V')}")
print("  ^ same Fontane column, different site bit.")


In [ ]:
#@title Feature encodings: weather { display-mode: "form" }
#@markdown §6 — VPD, rolling rainfall, and other per-day weather features.

def saturation_vp(temp_c):
    return 0.6108 * np.exp(17.27 * temp_c / (temp_c + 237.3))


def precompute_weather_features(weather_path):
    df = pd.read_excel(weather_path, header=10).iloc[1:].reset_index(drop=True)
    df["DAY"] = pd.to_datetime(df["DAY"])
    for c in ["IRRAD", "TMIN", "TMAX", "VAP", "WIND", "RAIN", "SNOWDEPTH"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.sort_values("DAY").reset_index(drop=True)
    for c in ["RAIN", "TMAX", "VAP", "IRRAD"]:
        df.loc[df[c] <= -990, c] = np.nan
    df["RAIN"] = df["RAIN"].fillna(0.0)
    df["VPD"] = (saturation_vp(df["TMAX"]) - df["VAP"]).clip(lower=0.0)
    df["RAIN_ROLL_7D"] = df["RAIN"].rolling(7, min_periods=1).sum()
    df["IRRAD_NORM"] = df["IRRAD"] / 1e4

    # 7-day means of VPD / TMAX / IRRAD. The LSTM maps weather to biomass
    # directly, so daily jitter would land in the output; the hybrid keeps the
    # raw series because the engine integrates weather into crop state (§13.1).
    df["VPD_7D"] = df["VPD"].rolling(7, min_periods=1).mean()
    df["TMAX_7D"] = df["TMAX"].rolling(7, min_periods=1).mean()
    df["IRRAD_NORM_7D"] = df["IRRAD_NORM"].rolling(7, min_periods=1).mean()

    features, features_smooth = {}, {}
    for _, row in df.iterrows():
        t = pd.Timestamp(row["DAY"]).normalize()
        features[t] = torch.tensor(
            [row["VPD"], row["TMAX"], row["RAIN_ROLL_7D"], row["IRRAD_NORM"]],
            dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device(),
        )
        features_smooth[t] = torch.tensor(
            [row["VPD_7D"], row["TMAX_7D"], row["RAIN_ROLL_7D"], row["IRRAD_NORM_7D"]],
            dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device(),
        )
    return features, features_smooth


_wf_L, _wf_L_smooth = precompute_weather_features(weather_paths["L"])
_wf_V, _wf_V_smooth = precompute_weather_features(weather_paths["V"])
WEATHER_FEATURES = {"L": _wf_L, "V": _wf_V}                        # hybrid: raw
WEATHER_FEATURES_SMOOTH = {"L": _wf_L_smooth, "V": _wf_V_smooth}   # LSTM: 7-day means
WEATHER_FEATURE_DIM = 4
print(f"Weather feature dim: {WEATHER_FEATURE_DIM} (raw + smoothed variants)")
print(f"Lelystad features pre-computed for {len(WEATHER_FEATURES['L'])} days")
print(f"Vredepeel features pre-computed for {len(WEATHER_FEATURES['V'])} days")


Each simulated day, WOFOST asks a small helper for the neural network’s inputs. It passes
the date, that day’s weather, and the current crop state — DVS, LAI and TAGP, all read live
from the engine’s shared variable store. The helper scales them to O(1) and returns one
combined feature vector.

Reading LAI and TAGP makes the network a **closed-loop controller**: those
quantities are integrals of assimilation, which `RFTRA` itself scales, so today's
inputs depend on yesterday's outputs. DVS does not — it is temperature-driven
and independent of the network.


In [ ]:
#@title Daily feature assembler { display-mode: "form" }
#@markdown §6 — combines weather, DVS/LAI/TAGP, and plot context for the stress NN.

# Crop state read from the engine each day. DVS locates the crop in its
# calendar; LAI tracks canopy size (transpiration scales with intercepted
# radiation); TAGP is added because the two decouple after anthesis.
CROP_STATE_VARS = ["DVS", "LAI", "TAGP"]

# Fixed, generous upper bounds — NOT fitted statistics, so the transform stays
# a constant a reader can check. Raw magnitudes span four orders of magnitude
# (DVS ~2, LAI ~5, TAGP ~2.4e4); feeding TAGP unscaled would swamp the first layer.
CROP_STATE_SCALES = torch.tensor(
    [2.0, 6.0, 25000.0],
    dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device(),
)

# Weather arrives in physical units. Scaling it is what makes reading LAI/TAGP
# mean anything. (Order: VPD, TMAX, RAIN_ROLL_7D, IRRAD_NORM.)
WEATHER_SCALES = torch.tensor(
    [5.0, 40.0, 160.0, 3.5],
    dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device(),
)


class PlotFeatureBuilder:
    """Daily feature assembler for a specific plot.

    Pre-computed weather features and the plot context tensor are baked in at
    construction; the crop state (DVS, LAI, TAGP) is read fresh from the kiosk
    on each call and scaled to O(1).

    LAI and TAGP are integrals of assimilation, which RFTRA scales, so they
    depend on earlier network outputs. The gradient therefore flows
    loss → LAI(t) → RFTRA(t−1) → weights. That coupling can run away
    (less LAI → more stress → less LAI), which is why the RFTRA ≈ 0.90
    initialisation in §9 matters.
    """

    def __init__(self, weather_features_by_date, plot_context):
        self.weather_features = weather_features_by_date
        self.plot_context = plot_context

    def __call__(self, day, drv, kiosk):
        wf = self.weather_features.get(pd.Timestamp(day).normalize())
        if wf is None:
            wf = torch.zeros(WEATHER_FEATURE_DIM, dtype=ComputeConfig.get_dtype(),
                             device=ComputeConfig.get_device())
        wf = wf / WEATHER_SCALES
        values = []
        for name in CROP_STATE_VARS:
            v = kiosk[name] if name in kiosk else torch.tensor(0.0)
            if not isinstance(v, torch.Tensor):
                v = torch.tensor(v, dtype=ComputeConfig.get_dtype(),
                                 device=ComputeConfig.get_device())
            values.append(v.flatten()[:1])
        crop_state = torch.cat(values) / CROP_STATE_SCALES
        return torch.cat([wf, crop_state, self.plot_context])


N_FEATURES = WEATHER_FEATURE_DIM + len(CROP_STATE_VARS) + PLOT_CONTEXT_DIM
print(f"Crop state read from engine: {CROP_STATE_VARS}")
print(f"Total NN input dim: {N_FEATURES}")


> ❓ **Discussion question**
>
> Crop stress is often cumulative: a week of low rain matters, not just today’s
> VPD. The stress NN gets same-day weather plus one explicit history term (7-
> day rolling rainfall) — and, through LAI and TAGP, the crop's accumulated
> growth so far. So it is not memoryless: the memory lives in the engine's state
> rather than in a hidden vector.
>
> Is that the right memory, though? LAI and TAGP integrate past growth. What
> drives drought stress is a soil water reservoir — rain in, evapotranspiration out
> — and `WaterbalancePP` tracks no such thing. `RAIN_ROLL_7D` is a crude
> stand-in: it weights day −7 exactly like today, drops day −8 to zero, and never
> saturates.
>
> What would you add? Longer or multiple rolling windows? A single learned
> scalar reservoir `S(t+1) = f(S(t), rain, ET)` — a differentiable bucket
> you could plot and sanity-check? Or a fully recurrent stress module, and what
> would that cost you in interpretability, given §11.4 and §11.5 read `RFTRA`
> as a lumped assimilate gate binned by DVS?


## 7. Engine setup

We use the standard `Wofost72_PP` stack — phenology, partitioning,
respiration, leaf dynamics, the works — but with a single targeted swap:
`evapotranspiration` → `NNStressFactor`. Everything else runs unchanged.


In [ ]:
#@title Engine setup
#@markdown §7 — WOFOST72_PP stack with `NNStressFactor` plugged into evapotranspiration.

crop_data_provider = YAMLCropDataProvider(fpath=crop_path, force_reload=True)
parameter_provider = ParameterProvider(
    cropdata=crop_data_provider,
    soildata=DummySoilDataProvider(),
    sitedata=WOFOST72SiteDataProvider(WAV=10),
)

OUTPUT_VARS = ["DVS", "LAI", "TAGP", "WLV", "TWST", "TWSO", "TWRT", "RFTRA"]
base_pcse_config = Configuration.from_pcse_config_file(conf_path)


def build_config(et_class=None, et_kwargs=None):
    cfg = Configuration(
        CROP=Wofost72, SOIL=WaterbalancePP,
        AGROMANAGEMENT=base_pcse_config.AGROMANAGEMENT,
        OUTPUT_VARS=OUTPUT_VARS.copy(),
        SUMMARY_OUTPUT_VARS=list(base_pcse_config.SUMMARY_OUTPUT_VARS),
        TERMINAL_OUTPUT_VARS=list(base_pcse_config.TERMINAL_OUTPUT_VARS),
        OUTPUT_INTERVAL=base_pcse_config.OUTPUT_INTERVAL,
        OUTPUT_INTERVAL_DAYS=base_pcse_config.OUTPUT_INTERVAL_DAYS,
        OUTPUT_WEEKDAY=base_pcse_config.OUTPUT_WEEKDAY,
        model_config_file=base_pcse_config.model_config_file,
        description=base_pcse_config.description,
    )
    if et_class is not None:
        comp = {"class": et_class}
        if et_kwargs:
            comp.update(et_kwargs)
        cfg.CROP_COMPONENTS = {"evapotranspiration": comp}
    return cfg


reference_config = build_config()     # default WOFOST72_PP, no NN

# One agromanagement file per (year, variety): the variety_name inside each file
# is what makes the engine call set_active_crop() and pull that cultivar's
# calibrated block, so this is the whole mechanism for per-cultivar physics.
agro_paths = {
    (year, variety): data_dir / "agro" / f"AGMT_{variety}_{year}.agro"
    for year in (2019, 2020)
    for variety in VARIETIES
}


def results_to_tensors(results, var_names=OUTPUT_VARS):
    out = {"day": [pd.Timestamp(r["day"]) for r in results]}
    for v in var_names:
        out[v] = torch.stack([
            torch.as_tensor(r[v], dtype=ComputeConfig.get_dtype(),
                            device=ComputeConfig.get_device())
            for r in results
        ])
    return out


# Cache weather data providers (one per site)
WEATHER_DATA_PROVIDERS = {
    site: ExcelWeatherDataProvider(str(weather_paths[site]))
    for site in ["L", "V"]
}


def run_engine(config, year, location, weather, variety):
    eng = Engine(config=config)
    eng.setup(
        copy.deepcopy(parameter_provider),
        weather,
        YAMLAgroManagementReader(agro_paths[(year, variety)]),
    )
    eng.run_till_terminate()
    return results_to_tensors(eng.get_output())


def run_plot_with_nn(plot, nn_model):
    feature_builder = PlotFeatureBuilder(
        WEATHER_FEATURES[plot.Location],
        make_plot_context_tensor(plot.Cultivar, plot.Nitrogen, plot.Irrigation, plot.Location),
    )
    cfg = build_config(NNStressFactor, et_kwargs={
        "nn_model": nn_model, "feature_builder": feature_builder,
    })
    return run_engine(
        cfg, plot.Year, plot.Location,
        WEATHER_DATA_PROVIDERS[plot.Location],
        CULTIVAR_TO_VARIETY[plot.Cultivar],
    )


def run_plot_reference(plot):
    return run_engine(
        reference_config, plot.Year, plot.Location,
        WEATHER_DATA_PROVIDERS[plot.Location],
        CULTIVAR_TO_VARIETY[plot.Cultivar],
    )


print("Engine helpers ready.")


## 8. Prepare for training

Three preparatory pieces: a loss function, a reference-WOFOST baseline run (to
normalise the loss), and a train/test split.


### 8.1 Loss function

A pooled normalised RMSE over the four observed variables (`WLV`, `TWST`,
`TWSO`, `LAI`) and `TWRT`. "Normalised" means dividing by the mean of the
observations, so each variable is on a comparable scale before weights are
applied.


In [ ]:
#@title Loss function helpers { display-mode: "form" }
#@markdown Pooled normalised RMSE over biomass/LAI observations.

OBS_TO_PCSE = {"LeavesDW": "WLV", "StemDW": "TWST", "tubersDW": "TWSO", "LAI": "LAI"}
ROOT_PCSE = "TWRT"
VAR_BASE_WEIGHTS = {"WLV": 1.0, "TWST": 1.0, "TWSO": 1.0, "LAI": 1.0, "TWRT": 0.5}


def get_plot_observations(plot, simulation_days):
    rows = obs_df[
        (obs_df["Year"] == plot.Year)
        & (obs_df["Location"] == plot.Location)
        & (obs_df["Plotnumber"] == plot.Plotnumber)
    ].copy()
    keep = rows[BIO_COLUMNS + [ROOT_COLUMN]].notna().any(axis=1)
    rows = rows.loc[keep].sort_values("Date").reset_index(drop=True)

    lookup = {pd.Timestamp(d).normalize(): i for i, d in enumerate(simulation_days)}
    last_idx = len(simulation_days) - 1
    last_day = pd.Timestamp(simulation_days[-1]).normalize()
    first_day = pd.Timestamp(simulation_days[0]).normalize()
    matched = []
    for _, r in rows.iterrows():
        key = pd.Timestamp(r["Date"]).normalize()
        if key in lookup:
            matched.append((lookup[key], r))
        elif key > last_day:
            matched.append((last_idx, r))
        elif key < first_day:
            matched.append((0, r))
    if not matched:
        return None, None
    indices = torch.tensor([m[0] for m in matched], dtype=torch.long)
    df_m = pd.DataFrame([m[1] for m in matched])
    targets = {p: torch.tensor(df_m[obs_col].to_numpy(),
                               dtype=ComputeConfig.get_dtype(),
                               device=ComputeConfig.get_device())
               for obs_col, p in OBS_TO_PCSE.items()}
    if df_m[ROOT_COLUMN].notna().any():
        targets[ROOT_PCSE] = torch.tensor(df_m[ROOT_COLUMN].to_numpy(),
                                          dtype=ComputeConfig.get_dtype(),
                                          device=ComputeConfig.get_device())
    return indices, targets


def per_plot_loss(results, targets, indices, weights):
    tl = torch.zeros((), dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device())
    diag = {}
    for name, target in targets.items():
        pred = results[name].index_select(0, indices)
        valid = torch.isfinite(target)
        if not torch.any(valid):
            continue
        pred = pred[valid]; target_valid = target[valid]
        scale = torch.mean(target_valid).abs().clamp_min(1e-6)
        rmse = torch.sqrt(torch.mean(((pred - target_valid) / scale) ** 2))
        tl = tl + weights[name] * rmse
        diag[name] = rmse.detach().cpu().item()
    return tl, diag


def pooled_loss(plots, runner, weights):
    total = torch.zeros((), dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device())
    n_used = 0
    all_diag = {k: [] for k in weights}
    per_plot = {}
    for p in plots:
        try:
            r = runner(p)
        except Exception as exc:
            print(f"  WARN: failed on plot {p}: {exc}")
            continue
        idx, tgt = get_plot_observations(p, r["day"])
        if idx is None:
            continue
        pl, diag = per_plot_loss(r, tgt, idx, weights)
        total = total + pl
        n_used += 1
        per_plot[(p.Year, p.Location, p.Plotnumber)] = r
        for k, v in diag.items():
            all_diag[k].append(v)
    avg_diag = {k: float(np.mean(v)) if v else float("nan") for k, v in all_diag.items()}
    return total / max(n_used, 1), avg_diag, per_plot, n_used


print("Loss machinery ready.")


### 8.2 Reference baseline + normalised weights

We first run the default WOFOST72_PP on every plot — no NN, no stress — to get a baseline
error per variable.

> **Why this baseline has to bound the observations.** `RFTRA` is a sigmoid in
> `[0, 1]`, so it can only pull growth *down* from potential production. That
> only works if PP sits *above* what the field achieved. If a plot out-yielded
> PP, no value of `RFTRA` can reach the observation — the target is unreachable
> by construction, not by bad luck in training.
>
> Per-cultivar parameters from `ajwdewit/WOFOST_crop_parameters` put the
> baseline in that regime for most plots: median obs/PP is about 0.9, with
> the largest deficit at N0. Fully fertilised N2 plots often sit near or
> above 1.0 — those targets are unreachable, but they are also where PP is
> already close, so they are not where a reduction factor has the most
> work.
>
> The baseline is also the loss normaliser. Tuber DM and LAI live on very
> different scales, so each term is rescaled by the reference RMSE.


In [ ]:
print(f"Running reference WOFOST72_PP on {len(PLOT_KEYS)} plot-years...")
ref_loss_uniform, ref_diag, REFERENCE_PLOT_RESULTS, n_ref_used = pooled_loss(
    PLOT_KEYS, run_plot_reference, VAR_BASE_WEIGHTS,
)
print(f"  Reference plots successfully simulated: {n_ref_used}/{len(PLOT_KEYS)}")
print(f"  Reference pooled loss (uniform weights): {ref_loss_uniform.item():.4f}")
print()
print("Per-variable RMSE (averaged across plots):")
for k, v in ref_diag.items():
    print(f"  {k:<6} {v:.4f}")

NORMALIZED_WEIGHTS = {
    k: VAR_BASE_WEIGHTS[k] / max(ref_diag[k], 0.1) for k in VAR_BASE_WEIGHTS
}
print()
print("Normalised loss weights (used during training):")
for k, v in NORMALIZED_WEIGHTS.items():
    print(f"  {k:<6} {v:.3f}")


### 8.3 Train/test split

How we partition the plot-years decides which kind of generalisation we measure. Set
`SPLIT_STRATEGY` in the cell below to switch between:

- **random** — a random 80/20 split of the plot-years. Tests interpolation within the design
  space: train and test share the same cultivars, sites, years, and treatments — just
  different plots.
- **year** — hold out a whole season (train on 2019, test on 2020). Tests extrapolation to
  an unseen weather year, a harder and more realistic ask.


In [ ]:
#@title Train/test split { display-mode: "form" }

# How to partition plot-years into train / test:
#   "random" - random 80/20 over all plot-years (interpolation within the
#              design space; train and test share cultivars, sites, years).
#   "year"   - hold out whole season(s); train on one year, test on another
#              (extrapolation to an unseen weather year).
SPLIT_STRATEGY = "year"     #@param ["random", "year"]
TEST_YEARS = (2020,)        # used only when SPLIT_STRATEGY == "year"


def split_random(plots, test_fraction=0.2, seed=42):
    """Random split of plot-years into train / test."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(plots))
    n_test = int(round(len(plots) * test_fraction))
    test_idx = set(idx[:n_test].tolist())
    train, test = [], []
    for i, p in enumerate(plots):
        (test if i in test_idx else train).append(p)
    return train, test


def split_by_year(plots, test_years=TEST_YEARS):
    """Hold out whole seasons: every plot whose Year is in test_years goes to
    the test set, the rest to train. Measures generalisation to an unseen
    weather year."""
    test_years = set(test_years)
    train = [p for p in plots if p.Year not in test_years]
    test = [p for p in plots if p.Year in test_years]
    return train, test


SPLIT_FUNCTIONS = {
    "random": split_random,
    "year": split_by_year,
}
if SPLIT_STRATEGY not in SPLIT_FUNCTIONS:
    raise ValueError(
        f"Unknown SPLIT_STRATEGY {SPLIT_STRATEGY!r}; "
        f"choose one of {sorted(SPLIT_FUNCTIONS)}"
    )

train_plots, test_plots = SPLIT_FUNCTIONS[SPLIT_STRATEGY](PLOT_KEYS)
if not train_plots or not test_plots:
    raise ValueError(
        f"Split {SPLIT_STRATEGY!r} produced an empty train or test set "
        f"(train={len(train_plots)}, test={len(test_plots)}); check the config."
    )

print(f"Split strategy: {SPLIT_STRATEGY}")
if SPLIT_STRATEGY == "year":
    print(f"  held-out test year(s): {sorted(set(TEST_YEARS))}")
print(f"  train plot-years: {len(train_plots)}")
print(f"  test plot-years: {len(test_plots)}")


## 9. Initialise the stress neural network

`StressNN` is a tiny MLP — 15 → 16 → 1 with a `SiLU` non-linearity and a
sigmoid output. With `init_no_stress=True`, the output bias is set so the
*untrained* network returns `RFTRA ≈ 0.90` (mild stress). This avoids two
pathologies:

- starting at `RFTRA = 0` would kill all biomass before the optimizer could
  learn anything;
- starting in the saturated tail of the sigmoid (e.g. `RFTRA = 0.99`) would
  shrink the gradient to almost nothing.

That is roughly 13× more gradient at init than a near-saturated sigmoid,
so the optimizer can actually move. It also matters for the closed loop
(less LAI → more stress → less LAI).


In [ ]:
torch.manual_seed(7)
stress_nn = StressNN(n_features=N_FEATURES, hidden_size=16, init_no_stress=True)
n_params = sum(p.numel() for p in stress_nn.parameters())
print(f"StressNN: {n_params} parameters")
print(stress_nn)


## 10. Train the model

Each training step does the following for every plot in the train set:

1. **Forward**: run the engine for the whole season. Every simulated day, the
   `NNStressFactor` component evaluates the NN to produce `RFTRA`, which
   reduces the day's gross assimilation.
2. **Compute loss**: extract simulated biomass at observation dates, compare to
   measurements, accumulate the normalised RMSE.
3. **Backward**: call `loss.backward()`. PyTorch propagates gradients through
   every operation in the engine — leaf growth, partitioning, root
   redistribution — and ends up at the NN parameters.
4. **Optimise**: Adam takes a step.

That's it. The engine looks like a normal PyTorch module from the optimizer's
point of view.

**Training is expensive** (each step = N engine runs × ~150 days each, a few
hours in total on CPU). This notebook ships pretrained weights in
[`pretrained/`](pretrained/) and loads them by default. Set
`FORCE_RETRAIN = True` to train from scratch.


In [ ]:
FORCE_RETRAIN = False
PRETRAINED_DIR = notebook_dir / "pretrained"
MODEL_DIR = data_temp_dir / "trained_models"
MODEL_DIR.mkdir(exist_ok=True)


def resolve_checkpoint(stem):
    """Shipped weights in pretrained/; retrains are written to data_temp."""
    shipped = PRETRAINED_DIR / f"{stem}_{SPLIT_STRATEGY}.pt"
    local = MODEL_DIR / f"{stem}_{SPLIT_STRATEGY}.pt"
    return shipped, local


shipped_model_path, model_path = resolve_checkpoint("stress_nn")

training_config = {
    "lr": 0.02,
    "max_steps": 60,
    "patience": 15,
    "min_delta": 5e-4,
}


def train_stress_nn(stress_nn, train_plots, test_plots, weights, cfg):
    optimizer = torch.optim.Adam(stress_nn.parameters(), lr=cfg["lr"])

    def runner(plot):
        return run_plot_with_nn(plot, stress_nn)

    train_history, test_history, diag_history = [], [], []
    best_loss, best_step = float("inf"), -1
    best_state = copy.deepcopy(stress_nn.state_dict())

    for step in range(cfg["max_steps"]):
        optimizer.zero_grad()
        train_loss, train_diag, _, _ = pooled_loss(train_plots, runner, weights)
        train_history.append(train_loss.detach().cpu().item())
        diag_history.append(train_diag)

        with torch.no_grad():
            test_loss, _, _, _ = pooled_loss(test_plots, runner, weights)
        test_history.append(test_loss.detach().cpu().item())

        if step % 2 == 0:
            print(f"  step {step:03d} | train={train_history[-1]:.4f} test={test_history[-1]:.4f}")

        if train_history[-1] < best_loss - cfg["min_delta"]:
            best_loss = train_history[-1]; best_step = step
            best_state = copy.deepcopy(stress_nn.state_dict())

        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(stress_nn.parameters(), max_norm=1.0)
        optimizer.step()

        if step - best_step >= cfg["patience"]:
            print(f"  early stopping at step {step}")
            break

    stress_nn.load_state_dict(best_state)
    return {
        "train_history": train_history,
        "test_history": test_history,
        "diag_history": diag_history,
    }


load_path = None
if not FORCE_RETRAIN:
    if model_path.exists():
        load_path = model_path
    elif shipped_model_path.exists():
        load_path = shipped_model_path

if load_path is not None:
    print(f"Loading saved model from {load_path}")
    saved = torch.load(load_path, weights_only=False)
    if saved.get("n_features") not in (None, N_FEATURES):
        raise RuntimeError(
            f"Checkpoint n_features={saved.get('n_features')} does not match "
            f"this notebook ({N_FEATURES}). Set FORCE_RETRAIN = True."
        )
    stress_nn.load_state_dict(saved["state_dict"])
    training_run = saved["training_run"]
    print(f"  Previously trained for {len(training_run['train_history'])} steps")
    print(f"  Saved train loss: {training_run['train_history'][-1]:.4f}")
    print(f"  Saved test loss: {training_run['test_history'][-1]:.4f}")
    print("  (Set FORCE_RETRAIN = True above to retrain.)")
else:
    print("Training from scratch — this will take a while...")
    training_run = train_stress_nn(stress_nn, train_plots, test_plots, NORMALIZED_WEIGHTS, training_config)
    torch.save({
        "state_dict": stress_nn.state_dict(),
        "training_run": training_run,
        "n_features": N_FEATURES,
    }, model_path)
    print(f"Saved trained model to {model_path}")

# Final evaluation — collect simulation outputs for every plot (train + test)
with torch.no_grad():
    final_train_loss, final_train_diag, train_results, _ = pooled_loss(
        train_plots, lambda p: run_plot_with_nn(p, stress_nn), NORMALIZED_WEIGHTS,
    )
    final_test_loss, final_test_diag, test_results, _ = pooled_loss(
        test_plots, lambda p: run_plot_with_nn(p, stress_nn), NORMALIZED_WEIGHTS,
    )

print()
print(f"FINAL: train loss={final_train_loss.item():.4f}            test loss={final_test_loss.item():.4f}")


## 11. Inspect model performance

We inspect five things:

1. **Loss curves** — did training converge? Does test track train?
2. **Per-plot fits** — does the model recover the shape of the trajectories?
3. **Learned stress factor per plot** — what does `RFTRA(t)` look like for a few plots?
4. **Stress profile by cultivar** — cultivar is mixed with site; treat as a residual view.
5. **Stress profile by nitrogen treatment** — does the gate rank the N label?


### 11.1 Loss curves


In [ ]:
#@title Plot: training loss curves { display-mode: "form" }
#@markdown §11.1 — train vs test loss over steps.

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(training_run["train_history"], label="train", linewidth=2)
ax.plot(training_run["test_history"], label="test", linewidth=2, linestyle="--")
ax.set_title(f"Pooled normalised loss ({SPLIT_STRATEGY} split)")
ax.set_xlabel("training step")
ax.set_ylabel("loss")
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()


### 11.2 Per-plot fits

For one plot per cultivar in the test set, we overlay:

- the default WOFOST72_PP simulation (solid),
- the trained hybrid simulation (dashed),
- the actual field observations (black dots).


In [ ]:
#@title Plot: per-plot model fits { display-mode: "form" }
#@markdown §11.2 — hybrid, WOFOST reference, and observations.

def _select_representative_plots(plots, n_per_cultivar=1):
    by_c = {}
    for p in plots:
        by_c.setdefault(p.Cultivar, []).append(p)
    selected = []
    for c in CULTIVARS:
        if c in by_c:
            selected.extend(by_c[c][:n_per_cultivar])
    return selected


display_plots = _select_representative_plots(test_plots, n_per_cultivar=1)
if not display_plots:
    display_plots = _select_representative_plots(train_plots, n_per_cultivar=1)
print(f"Showing fits for {len(display_plots)} plots (one per cultivar in the test set)")


def _plot_observed(ax, plot, obs_col):
    sub = obs_df[
        (obs_df["Year"] == plot.Year)
        & (obs_df["Location"] == plot.Location)
        & (obs_df["Plotnumber"] == plot.Plotnumber)
    ]
    ax.scatter(sub["Date"], sub[obs_col], s=28, color="black", zorder=5, label="Observed")


fig, axes = plt.subplots(len(display_plots), 4, figsize=(20, 3 * len(display_plots)),
                          squeeze=False)
PLOT_VARS = [("WLV", "LeavesDW", "Leaf DM"),
             ("TWST", "StemDW", "Stem DM"),
             ("TWSO", "tubersDW", "Tuber DM"),
             ("LAI", "LAI", "LAI")]
for row, plot in enumerate(display_plots):
    key = (plot.Year, plot.Location, plot.Plotnumber)
    fitted = test_results.get(key) or train_results.get(key)
    ref = REFERENCE_PLOT_RESULTS.get(key)
    label = f"{plot.Cultivar}@{plot.Location}, {plot.Nitrogen}, {plot.Irrigation}, {plot.Year}"
    for col, (var, obs_col, title) in enumerate(PLOT_VARS):
        ax = axes[row, col]
        if ref is not None:
            ax.plot(ref["day"], ref[var].detach().cpu().numpy(),
                    label="WOFOST72_PP", linewidth=1.6)
        if fitted is not None:
            ax.plot(fitted["day"], fitted[var].detach().cpu().numpy(),
                    label="Hybrid (NN stress)", linewidth=1.6, linestyle="--")
        _plot_observed(ax, plot, obs_col)
        ax.set_title(f"{title} — {label}", fontsize=9)
        ax.grid(alpha=0.3); ax.tick_params(axis="x", rotation=30, labelsize=7)
axes[0, 0].legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()


### 11.3 Learned stress factor over the season

`RFTRA(t)` is what the NN actually produces. A value of 1.0 means "no stress",
a value of 0.0 means "complete shutdown of carbon fixation". Reasonable
trajectories should sit between, dipping during the parts of the season where
the model needs to reduce biomass to match observations.


In [ ]:
#@title Plot: learned RFTRA trajectories { display-mode: "form" }
#@markdown §11.3 — daily stress factor over the season.

fig, axes = plt.subplots(1, len(display_plots), figsize=(4 * len(display_plots), 4),
                          squeeze=False)
for col, plot in enumerate(display_plots):
    ax = axes[0, col]
    key = (plot.Year, plot.Location, plot.Plotnumber)
    fitted = test_results.get(key) or train_results.get(key)
    if fitted is not None:
        ax.plot(fitted["day"], fitted["RFTRA"].detach().cpu().numpy(),
                color="tab:orange", linewidth=2)
    ax.axhline(1.0, color="black", linestyle=":", alpha=0.5, label="no stress")
    ax.set_title(f"{plot.Cultivar}@{plot.Location} {plot.Nitrogen}{plot.Irrigation} {plot.Year}", fontsize=9)
    ax.set_ylabel("RFTRA (learned stress factor)")
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.3); ax.tick_params(axis="x", rotation=30, labelsize=7)
plt.tight_layout(); plt.show()


### 11.4 Stress profile by cultivar

Per-plot trajectories show one learned curve at a time. Grouping by cultivar is
a natural next view, with a hard limit: **cultivar is confounded with site**
(C1/C2/C3 only at Lelystad, C4/C5/C6 only at Vredepeel; section 3). A curve
that separates by C-code also separates by soil and weather station.

The plot is still worth drawing. Cultivar genetics already live in the
physics (`TSUM1` / `TSUM2` / `SPAN` / `RGRLAI` and partitioning), so `RFTRA`
is a residual on carbon gain, not a stand-in for phenology. We collapse
calendar dates to DVS, bin `RFTRA`, and average across plots. Read every
curve as **cultivar-and-site**. The cleaner treatment axis is nitrogen (§11.5).


In [ ]:
#@title Plot: RFTRA profile by cultivar { display-mode: "form" }
#@markdown §11.4 — mean RFTRA binned by DVS.

DVS_BINS = np.linspace(0.0, 2.0, 21)
DVS_CENTERS = 0.5 * (DVS_BINS[:-1] + DVS_BINS[1:])


def aggregate_rftra_by_dvs(plots, results_lookups, group_key):
    """Bin RFTRA by DVS, averaging across plots that share `group_key(plot)`.

    `results_lookups` is an iterable of dicts to try in order (e.g. test_results
    then train_results) — the first one containing the plot wins.
    Returns {group_value: rftra_means_array_of_len_DVS_CENTERS}.
    """
    groups = sorted({group_key(p) for p in plots})
    sums = {g: np.zeros(len(DVS_CENTERS)) for g in groups}
    counts = {g: np.zeros(len(DVS_CENTERS)) for g in groups}
    for plot in plots:
        key = (plot.Year, plot.Location, plot.Plotnumber)
        r = None
        for d in results_lookups:
            if key in d:
                r = d[key]; break
        if r is None:
            continue
        dvs = r["DVS"].detach().cpu().numpy()
        rftra = r["RFTRA"].detach().cpu().numpy()
        active = (rftra > 1e-6) & (dvs > 0)
        if not active.any():
            continue
        bin_idx = np.clip(np.digitize(dvs[active], DVS_BINS) - 1, 0, len(DVS_CENTERS) - 1)
        g = group_key(plot)
        for b, v in zip(bin_idx, rftra[active]):
            sums[g][b] += v
            counts[g][b] += 1
    out = {}
    for g in groups:
        means = np.full(len(DVS_CENTERS), np.nan)
        nz = counts[g] > 0
        means[nz] = sums[g][nz] / counts[g][nz]
        out[g] = means
    return out


cultivar_rftra_by_dvs = aggregate_rftra_by_dvs(
    train_plots + test_plots,
    [train_results, test_results],
    lambda p: p.Cultivar,
)

fig, ax = plt.subplots(figsize=(11, 5))
for i, c in enumerate(CULTIVARS):
    curve = cultivar_rftra_by_dvs.get(c)
    if curve is not None and not np.isnan(curve).all():
        ax.plot(DVS_CENTERS, curve, marker="o", linewidth=2,
                label=c, color=plt.cm.tab10(i))
ax.axhline(1.0, color="black", linestyle=":", alpha=0.6)
ax.axvline(1.0, color="red", linestyle=":", alpha=0.6, label="DVS=1 (anthesis)")
ax.set_xlabel("DVS (development stage)")
ax.set_ylabel("mean learned RFTRA")
ax.set_title("Seasonal stress profile per cultivar")
ax.set_ylim(0.0, 1.05); ax.grid(alpha=0.3); ax.legend(loc="lower left", fontsize=9)
plt.tight_layout(); plt.show()


### 11.5 Stress profile by nitrogen treatment

Same DVS-binned aggregation as 11.4, but split by **N level** (N0 / N1 / N2)
instead of cultivar — pooled over all cultivars, sites, and W treatments. The
nitrogen treatment is encoded as a scalar input to the NN (N0=0.0, N1=0.3,
N2=1.3), so this plot is the cleanest test of whether the NN actually uses
that input.

If the N label is used, the ranking should be:

- **N0** (no fertiliser) — strongest reduction, lowest RFTRA;
- **N1** (~30% advised) — intermediate;
- **N2** (~130% advised) — weakest reduction, RFTRA closest to 1.

If the three curves come out essentially overlapping, the NN didn't
differentiate by N treatment — it absorbed the N-related residual into other
features (cultivar, weather) or just didn't fit it.

A separation here means the gate **ranks the N label**. It does not mean the
network learned nitrogen physiology: `RFTRA` is still a lumped assimilate
reduction (section 5), and N is an input code, not a fertiliser flux.


In [ ]:
#@title Plot: RFTRA profile by N treatment { display-mode: "form" }
#@markdown §11.5 — mean RFTRA binned by DVS and N level.

N_LEVELS = ["N0", "N1", "N2"]
n_rftra_by_dvs = aggregate_rftra_by_dvs(
    train_plots + test_plots,
    [train_results, test_results],
    lambda p: p.Nitrogen,
)

n_colors = {"N0": "tab:red", "N1": "tab:orange", "N2": "tab:green"}
n_labels = {
    "N0": "N0 (no fertiliser)",
    "N1": "N1 (30% advised)",
    "N2": "N2 (130% advised)",
}

fig, ax = plt.subplots(figsize=(11, 5))
for n in N_LEVELS:
    curve = n_rftra_by_dvs.get(n)
    if curve is not None and not np.isnan(curve).all():
        ax.plot(DVS_CENTERS, curve, marker="o", linewidth=2,
                color=n_colors[n], label=n_labels[n])
ax.axhline(1.0, color="black", linestyle=":", alpha=0.6)
ax.axvline(1.0, color="red", linestyle=":", alpha=0.6, label="DVS=1 (anthesis)")
ax.set_xlabel("DVS (development stage)")
ax.set_ylabel("mean learned RFTRA")
ax.set_title("Seasonal stress profile per nitrogen treatment")
ax.set_ylim(0.0, 1.05); ax.grid(alpha=0.3); ax.legend(loc="lower left", fontsize=9)
plt.tight_layout(); plt.show()

print("Mean RFTRA across the active season (DVS > 0) per N treatment:")
for n in N_LEVELS:
    finite = n_rftra_by_dvs[n][~np.isnan(n_rftra_by_dvs[n])]
    if len(finite):
        print(f"  {n}: mean = {finite.mean():.3f}  min = {finite.min():.3f}  max = {finite.max():.3f}")


> ❓ **Discuss with a neighbour**
>
> Compare the two profile plots
> above. Which axis (cultivar-and-site / N treatment) shows the cleaner
> separation? If N is cleaner, is that physiology — or a treatment label
> entering a lumped assimilate gate that cannot tell water from nitrogen?
>


## 12. Reference comparison: default WOFOST

Below we line up the trained hybrid against potential-production WOFOST72_PP on per-
variable normalised RMSE. Before reading the bars, please keep the framing in mind:

> ⚠️ **This is not an "is hybrid better than WOFOST?" benchmark.**
> WOFOST runs on maintained per-cultivar parameters, but two things are still
> unmatched: sowing is a generic 20 April for every plot, and the simulated
> maturity date can miss the recorded harvest depending on the site-year. Treat
> the bars as a diagnostic, not a leaderboard.
>
> **A note on what the physics here can and cannot do.** diffwofost
> implements WOFOST 7.2 potential production only: there is no NPK module
> and no `WaterbalanceFD`, so `Wofost72_WLP` is not available. That is not an
> oversight to route around — it is the experiment. The NN learns a lumped
> reduction of carbon gain (`GASS = PGASS × RFTRA`) because water and
> nitrogen limitation are missing from the engine. It does not identify
> either process separately.
>
> One structural limit is worth naming. `RFTRA` was built as a transpiration
> reduction. Nitrogen in WOFOST 8.1 acts through AMAX and accelerated leaf
> senescence. So the N ranking in 11.5 is a treatment label in that lumped
> gate, not N physiology.

A train/test gap also matters here: small = good generalisation; large = overfitting on the
training plots.


In [ ]:
#@title Plot: hybrid vs WOFOST RMSE { display-mode: "form" }
#@markdown §12 — per-variable normalised RMSE bars.

def per_plot_diag(plots, results_dict):
    out = {k: [] for k in NORMALIZED_WEIGHTS}
    for plot in plots:
        key = (plot.Year, plot.Location, plot.Plotnumber)
        r = results_dict.get(key)
        if r is None:
            continue
        idx, tgt = get_plot_observations(plot, r["day"])
        if idx is None:
            continue
        for name, target in tgt.items():
            pred = r[name].index_select(0, idx)
            valid = torch.isfinite(target)
            if not torch.any(valid):
                continue
            pred = pred[valid]; target_valid = target[valid]
            scale = torch.mean(target_valid).abs().clamp_min(1e-6)
            rmse = torch.sqrt(torch.mean(((pred - target_valid) / scale) ** 2))
            out[name].append(rmse.item())
    return out


train_diag_per_plot = per_plot_diag(train_plots, train_results)
test_diag_per_plot = per_plot_diag(test_plots, test_results)
ref_diag_per_plot = per_plot_diag(train_plots + test_plots, REFERENCE_PLOT_RESULTS)

variables = list(NORMALIZED_WEIGHTS.keys())
n_vars = len(variables)
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(n_vars)
width = 0.27

train_means = [np.mean(train_diag_per_plot[v]) if train_diag_per_plot[v] else 0 for v in variables]
test_means = [np.mean(test_diag_per_plot[v]) if test_diag_per_plot[v] else 0 for v in variables]
ref_means = [np.mean(ref_diag_per_plot[v]) if ref_diag_per_plot[v] else 0 for v in variables]

ax.bar(x - width, ref_means, width, label="WOFOST72_PP",
       color="lightgray", edgecolor="black")
ax.bar(x, train_means, width, label="Hybrid, on TRAIN", color="steelblue", edgecolor="black")
ax.bar(x + width, test_means, width, label="Hybrid, on TEST", color="orange", edgecolor="black")

ax.set_xticks(x); ax.set_xticklabels(variables)
ax.set_ylabel("normalised RMSE (averaged over plots)")
ax.set_title("Per-variable RMSE: WOFOST72_PP vs. trained hybrid")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

print()
print("Per-variable summary:")
for v, ref, tr, te in zip(variables, ref_means, train_means, test_means):
    print(f"  {v:<6} default={ref:.3f}  train={tr:.3f}  test={te:.3f}  "
          f"(test/train: {te / max(tr, 1e-6):.2f})")


> ❓ **Discuss with a neighbour**
>
> Which variables does the hybrid
> close the gap on most? Which does it barely move? Where do you think the
> remaining test-set error comes from — the NN's capacity, the feature set, or
> sowing and phenology that this setup does not calibrate and the NN cannot reach?
>


## 13. Comparison vs. a pure-ML LSTM

To complete the picture we train a **pure data-driven model with no physics in
the loop**: a small LSTM that maps per-day features → daily organ biomass. This
tells us how much of the hybrid's fit comes from the engine versus the data.

We use an LSTM (not just an MLP) because the hybrid already gets weather history
through the engine's state (`LAI`, `TAGP`, `DVS`). A fair pure-ML reference needs
its own way to integrate past weather; that is what the hidden state does.

**Architecture:**

```
Per-day input (13):
  - Weather:    7-day means of VPD, TMAX, IRRAD + 7-day rolling rain   [4]
  - Time:       days_since_sowing / 200                               [1]
  - Treatment:  site, N level, W level, variety one-hot (5)           [8]

       inputs (200, 13)
              │
              ▼
       LSTM (16 hidden, 1 layer)
              │
              ▼
       Linear → daily increment (softplus on TWSO: tubers don't shrink)
              │
              ▼
       cumsum over time  →  4 organ trajectories (WLV, TWST, TWSO, LAI)
```

**On the inductive bias.** The LSTM emits a daily *rate* and integrates it,
rather than an absolute level per day. With only ~7 of 200 days carrying a
gradient, a per-day level is almost unconstrained: trajectories jump, and tubers
can appear in April. Integrating a rate is the same structural assumption WOFOST
makes — state is the integral of a rate — so the comparison is about missing
physics, not a broken baseline. A softplus on the TWSO increment keeps tuber
mass from shrinking.

> ⚠️ **Same caveat as the WOFOST baseline.** This is a fair reference
> architecture, not a tuned state-of-the-art. Read the comparison qualitatively:
> how stable, how data-hungry, how prone to overfitting is a black-box model
> versus a hybrid one at this sample size? With the temporal split (train 2019,
> test 2020) the LSTM must extrapolate to an unseen weather year from 84
> training plots — which is the honest version of the question.


### 13.1 LSTM data preparation

Pre-build the per-day feature sequences (one per plot) and the
observation-index / target tensors. Doing this once up front means the training
loop just slices into prebuilt tensors.


In [ ]:
#@title LSTM data preparation { display-mode: "form" }
#@markdown Pre-build per-plot feature sequences and observation targets for the pure-ML baseline.

# Order-of-magnitude max we'd expect per organ over a whole season.
PURE_OUTPUT_SCALES = torch.tensor(
    [3000.0, 2500.0, 20000.0, 8.0],   # WLV, TWST, TWSO, LAI
    dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device(),
)

SOWING_DATE_BY_YEAR = {
    2019: pd.Timestamp("2019-04-20"),
    2020: pd.Timestamp("2020-04-20"),
}
DAYS_NORMALIZER = 200.0

PURE_OBS_VARS = ["LeavesDW", "StemDW", "tubersDW", "LAI"]
PURE_OBS_TO_PCSE = ["WLV", "TWST", "TWSO", "LAI"]
PURE_N_FEATURES = WEATHER_FEATURE_DIM + 1 + PLOT_CONTEXT_DIM


def _raw_pure_features(plot, date):
    norm_date = pd.Timestamp(date).normalize()
    # Smoothed weather: the LSTM maps weather to biomass directly, so daily
    # jitter in VPD/TMAX/IRRAD would land in the output. The hybrid uses raw
    # weather; the engine integrates it into crop state.
    wf = WEATHER_FEATURES_SMOOTH[plot.Location].get(norm_date)
    if wf is None:
        wf = torch.zeros(WEATHER_FEATURE_DIM, dtype=ComputeConfig.get_dtype(),
                         device=ComputeConfig.get_device())
    days = (norm_date - SOWING_DATE_BY_YEAR[plot.Year]).days
    days_t = torch.tensor(
        [days / DAYS_NORMALIZER],
        dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device(),
    )
    ctx = make_plot_context_tensor(plot.Cultivar, plot.Nitrogen, plot.Irrigation, plot.Location)
    return torch.cat([wf, days_t, ctx])


def pure_features_for_date(plot, date):
    raw = _raw_pure_features(plot, date)
    if "PURE_FEATURES_MEAN" in globals() and "PURE_FEATURES_STD" in globals():
        return (raw - PURE_FEATURES_MEAN) / PURE_FEATURES_STD
    return raw


def _build_raw(plots):
    feats = []
    for plot in plots:
        rows = obs_df[
            (obs_df["Year"] == plot.Year)
            & (obs_df["Location"] == plot.Location)
            & (obs_df["Plotnumber"] == plot.Plotnumber)
            & obs_df[PURE_OBS_VARS].notna().any(axis=1)
        ]
        for _, r in rows.iterrows():
            feats.append(_raw_pure_features(plot, r["Date"]))
    return torch.stack(feats) if feats else None


_raw_train = _build_raw(train_plots)
PURE_FEATURES_MEAN = _raw_train.mean(dim=0)
PURE_FEATURES_STD = _raw_train.std(dim=0).clamp_min(1e-3)
# Keep one-hots / binary flags in their raw {0, 1} scale.
PURE_FEATURES_STD[WEATHER_FEATURE_DIM + 1:] = 1.0
PURE_FEATURES_MEAN[WEATHER_FEATURE_DIM + 1:] = 0.0

PURE_VARIABLE_WEIGHTS = torch.tensor(
    [NORMALIZED_WEIGHTS[v] for v in PURE_OBS_TO_PCSE],
    dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device(),
)


LSTM_SEQ_LEN = 200    # days from sowing
LSTM_HIDDEN = 16

# Per-day increment scale: season-total magnitude divided by season length.
PURE_RATE_SCALES = PURE_OUTPUT_SCALES / LSTM_SEQ_LEN


class PureLSTM(torch.nn.Module):
    """Sequence-aware pure-ML reference. (seq_len, n_features) → (seq_len, 4).

    Emits a daily **increment** and integrates it, rather than emitting an
    absolute level per day. That is the one structural thing WOFOST gets for
    free: state is the integral of a rate.

    TWSO additionally gets a softplus on its increment: tubers do not shrink.
    WLV/TWST/LAI stay signed — they rise, then senesce.
    """

    def __init__(self, n_features, hidden=LSTM_HIDDEN, n_outputs=4, rate_scales=None):
        super().__init__()
        self.lstm = torch.nn.LSTM(
            input_size=n_features, hidden_size=hidden,
            num_layers=1, batch_first=True,
        )
        self.head = torch.nn.Linear(hidden, n_outputs)
        if rate_scales is None:
            rate_scales = PURE_RATE_SCALES
        self.register_buffer("rate_scales", rate_scales.clone().to(
            dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device(),
        ))
        self.to(device=ComputeConfig.get_device(), dtype=ComputeConfig.get_dtype())

    def forward(self, seq):
        if seq.dim() == 2:
            seq = seq.unsqueeze(0)
        h, _ = self.lstm(seq)                       # (B, T, hidden)
        raw = self.head(h)                          # (B, T, 4) daily rates
        # index 2 is TWSO — force its increment non-negative.
        delta = torch.cat([
            raw[..., 0:2],
            torch.nn.functional.softplus(raw[..., 2:3]),
            raw[..., 3:4],
        ], dim=-1) * self.rate_scales
        # Euler integration from zero: state[t] is the sum of the increments
        # *before* t, so day 0 is exactly zero — nothing exists at sowing.
        cum = torch.cumsum(delta, dim=1)
        out = torch.cat([torch.zeros_like(cum[:, :1]), cum[:, :-1]], dim=1)
        return out.clamp(min=0.0)   # negative biomass is meaningless


def build_lstm_data(plot, n_days=LSTM_SEQ_LEN):
    sowing = SOWING_DATE_BY_YEAR[plot.Year]
    feats = []
    for d in range(n_days):
        date = sowing + pd.Timedelta(days=d)
        feats.append(pure_features_for_date(plot, date))
    seq = torch.stack(feats)

    rows = obs_df[
        (obs_df["Year"] == plot.Year)
        & (obs_df["Location"] == plot.Location)
        & (obs_df["Plotnumber"] == plot.Plotnumber)
        & obs_df[PURE_OBS_VARS].notna().any(axis=1)
    ].sort_values("Date").reset_index(drop=True)
    if rows.empty:
        return seq, None, None
    indices, targets = [], []
    for _, r in rows.iterrows():
        d = (pd.Timestamp(r["Date"]).normalize() - sowing).days
        d = max(0, min(d, n_days - 1))
        indices.append(d)
        targets.append([r[v] for v in PURE_OBS_VARS])
    idx_t = torch.tensor(indices, dtype=torch.long, device=ComputeConfig.get_device())
    tgt_t = torch.tensor(targets, dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device())
    return seq, idx_t, tgt_t


print("Pre-building LSTM sequences for every plot...")
lstm_data_train = {p: build_lstm_data(p) for p in train_plots}
lstm_data_test = {p: build_lstm_data(p) for p in test_plots}
print(f"  train sequences: {len(lstm_data_train)} plots, each ({LSTM_SEQ_LEN}, {PURE_N_FEATURES})")
print(f"  test  sequences: {len(lstm_data_test)} plots, each ({LSTM_SEQ_LEN}, {PURE_N_FEATURES})")


### 13.2 Train (or load) the LSTM

The LSTM trains the same way as the hybrid — pooled normalised RMSE, Adam,
early stopping. To prevent overfitting on the small dataset, we (a) carve a
separate validation split out of the training plots so early stopping does not
peek at the test set, and (b) add a touch of weight decay. As with the hybrid,
we load a pre-trained checkpoint by default.


In [ ]:
#@title Train LSTM { display-mode: "form" }
shipped_lstm_path, lstm_model_path = resolve_checkpoint("pure_lstm")


def lstm_pooled_loss(lstm, plot_data_dict, weights_tensor):
    var_total = torch.zeros(4, dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device())
    var_count = torch.zeros(4, dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device())
    plot_preds = {}
    for plot, (seq, idx, tgt) in plot_data_dict.items():
        if idx is None or len(idx) == 0:
            continue
        pred_full = lstm(seq)
        pred_at_obs = pred_full[0, idx, :]
        plot_preds[plot] = pred_full[0]
        for i in range(4):
            valid = torch.isfinite(tgt[:, i])
            if not valid.any():
                continue
            pv = pred_at_obs[valid, i]
            tv = tgt[valid, i]
            scale = tv.mean().abs().clamp_min(1e-6)
            rmse = torch.sqrt(torch.mean(((pv - tv) / scale) ** 2))
            var_total[i] = var_total[i] + weights_tensor[i] * rmse
            var_count[i] = var_count[i] + 1
    n_plots = sum(1 for _, (_, idx, _) in plot_data_dict.items() if idx is not None)
    if n_plots == 0:
        return torch.zeros((), dtype=ComputeConfig.get_dtype()), {}, {}
    total = var_total.sum() / n_plots
    diag = {
        name: (var_total[i] / var_count[i].clamp_min(1)).item() / weights_tensor[i].item()
        for i, name in enumerate(PURE_OBS_TO_PCSE)
    }
    return total, diag, plot_preds


torch.manual_seed(23)
pure_lstm = PureLSTM(n_features=PURE_N_FEATURES, hidden=LSTM_HIDDEN)
lstm_n_params = sum(p.numel() for p in pure_lstm.parameters())
print(f"PureLSTM: {lstm_n_params} parameters")

lstm_load_path = None
if not FORCE_RETRAIN:
    if lstm_model_path.exists():
        lstm_load_path = lstm_model_path
    elif shipped_lstm_path.exists():
        lstm_load_path = shipped_lstm_path

if lstm_load_path is not None:
    saved = torch.load(lstm_load_path, weights_only=False)
    pure_lstm.load_state_dict(saved["state_dict"])
    lstm_run = saved["lstm_run"]
    print(f"Loaded saved LSTM from {lstm_load_path}")
    print(f"  Saved train loss: {lstm_run['train_history'][-1]:.4f}")
    print(f"  Saved test loss: {lstm_run['test_history'][-1]:.4f}")
else:
    LSTM_VAL_FRACTION = 0.2
    LSTM_VAL_SEED = 1234
    _val_rng = np.random.default_rng(LSTM_VAL_SEED)
    _all_train_plots_for_lstm = list(lstm_data_train.keys())
    _shuffled = list(_all_train_plots_for_lstm)
    _val_rng.shuffle(_shuffled)
    _n_val = max(1, int(round(len(_shuffled) * LSTM_VAL_FRACTION)))
    _val_plots = _shuffled[:_n_val]
    _train_subset_plots = _shuffled[_n_val:]
    lstm_data_train_subset = {p: lstm_data_train[p] for p in _train_subset_plots}
    lstm_data_val = {p: lstm_data_train[p] for p in _val_plots}
    print(f"  LSTM split: train_subset={len(lstm_data_train_subset)} "
          f"val={len(lstm_data_val)} test={len(lstm_data_test)}")

    optimizer = torch.optim.Adam(pure_lstm.parameters(), lr=0.005, weight_decay=1e-3)
    lstm_run = {"train_history": [], "val_history": [], "test_history": [], "diag_history": []}

    LSTM_MAX_STEPS, LSTM_PATIENCE, LSTM_MIN_DELTA = 1500, 100, 1e-3
    best_val_loss, best_step = float("inf"), -1
    best_state = copy.deepcopy(pure_lstm.state_dict())

    for step in range(LSTM_MAX_STEPS):
        optimizer.zero_grad()
        train_loss, train_diag, _ = lstm_pooled_loss(pure_lstm, lstm_data_train_subset, PURE_VARIABLE_WEIGHTS)
        with torch.no_grad():
            val_loss, _, _ = lstm_pooled_loss(pure_lstm, lstm_data_val, PURE_VARIABLE_WEIGHTS)
            test_loss, _, _ = lstm_pooled_loss(pure_lstm, lstm_data_test, PURE_VARIABLE_WEIGHTS)
        lstm_run["train_history"].append(train_loss.item())
        lstm_run["val_history"].append(val_loss.item())
        lstm_run["test_history"].append(test_loss.item())
        lstm_run["diag_history"].append(train_diag)

        if step % 50 == 0:
            marker = " *" if val_loss.item() < best_val_loss - LSTM_MIN_DELTA else ""
            print(f"  pure_lstm step {step:04d} | train={train_loss.item():.4f} "
                  f"val={val_loss.item():.4f}{marker} test={test_loss.item():.4f}")

        if val_loss.item() < best_val_loss - LSTM_MIN_DELTA:
            best_val_loss = val_loss.item(); best_step = step
            best_state = copy.deepcopy(pure_lstm.state_dict())

        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(pure_lstm.parameters(), max_norm=2.0)
        optimizer.step()

        if step - best_step >= LSTM_PATIENCE:
            print(f"  pure_lstm early stopping at step {step} (best val {best_val_loss:.4f})")
            break

    pure_lstm.load_state_dict(best_state)
    torch.save({
        "state_dict": pure_lstm.state_dict(),
        "lstm_run": lstm_run,
        "n_features": PURE_N_FEATURES,
        "seq_len": LSTM_SEQ_LEN,
        "hidden": LSTM_HIDDEN,
    }, lstm_model_path)
    print(f"Saved trained LSTM to {lstm_model_path}")

with torch.no_grad():
    final_lstm_train_loss, final_lstm_train_diag, lstm_train_preds = lstm_pooled_loss(
        pure_lstm, lstm_data_train, PURE_VARIABLE_WEIGHTS,
    )
    final_lstm_test_loss, final_lstm_test_diag, lstm_test_preds = lstm_pooled_loss(
        pure_lstm, lstm_data_test, PURE_VARIABLE_WEIGHTS,
    )

print()
print(f"Pure LSTM | final train loss = {final_lstm_train_loss.item():.4f}")
print(f"Pure LSTM | final test  loss = {final_lstm_test_loss.item():.4f}")


### 13.3 Three-way comparison

Three views, each telling a different story.

**(i) Per-variable test RMSE.** Potential-production WOFOST72_PP, hybrid,
and pure LSTM side by side on the held-out test set. The hybrid typically
lands in between — it doesn't have the LSTM's raw memorisation capacity
but it generalises more reliably (see view (iii)).


In [ ]:
#@title Plot: three-model test RMSE { display-mode: "form" }
#@markdown §13 — per-variable bars on the test set.

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(PURE_OBS_TO_PCSE))
width = 0.27
ax.bar(x - width, [ref_diag[v] for v in PURE_OBS_TO_PCSE], width,
       label="WOFOST72_PP", color="lightgray", edgecolor="black")
ax.bar(x, [final_test_diag.get(v, np.nan) for v in PURE_OBS_TO_PCSE], width,
       label="Hybrid (PP + NN stress)", color="tab:orange", edgecolor="black")
ax.bar(x + width, [final_lstm_test_diag.get(v, np.nan) for v in PURE_OBS_TO_PCSE], width,
       label="Pure LSTM (no physics)", color="tab:purple", edgecolor="black")
ax.set_xticks(x); ax.set_xticklabels(PURE_OBS_TO_PCSE)
ax.set_ylabel("normalised RMSE on TEST set")
ax.set_title(f"Per-variable test RMSE: three models, {SPLIT_STRATEGY} split")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()


**(ii) Trajectories on single plots.** We show all three models on one train plot and one test
plot. Overlaying them on the same plot makes the qualitative differences clear: the hybrid's
trajectories obey the engine's physics (smooth, physiologically plausible curves with the right
shape), while the LSTM produces whatever shape minimises its loss — sometimes
biologically odd kinks or wiggles, particularly near the start of the season where it has little
context. Comparing the train plot against the held-out test plot also hints at how much each
model leans on having seen that plot during training.


In [ ]:
#@title Plot: three-model trajectories { display-mode: "form" }
#@markdown §13 — one train plot and one test plot, all three models.

ORGAN_LIST_LSTM = [
    ("WLV", "LeavesDW", "Leaves (WLV)", 0),
    ("TWST", "StemDW", "Stems (TWST)", 1),
    ("TWSO", "tubersDW", "Storage organs (TWSO)", 2),
    ("LAI", "LAI", "LAI", 3),
]


def plot_three_models(sample_plot, split_name):
    """Overlay default WOFOST, hybrid, and pure LSTM against observations for
    a single plot. split_name ("TRAIN"/"TEST") is shown in the title."""
    key = (sample_plot.Year, sample_plot.Location, sample_plot.Plotnumber)

    sample_data = lstm_data_test.get(sample_plot) or lstm_data_train.get(sample_plot)
    if sample_data is None:
        seq_sample, _, _ = build_lstm_data(sample_plot)
    else:
        seq_sample = sample_data[0]
    with torch.no_grad():
        lstm_traj = pure_lstm(seq_sample)[0].cpu().numpy()
    sowing_for_sample = SOWING_DATE_BY_YEAR[sample_plot.Year]
    lstm_dates = [sowing_for_sample + pd.Timedelta(days=d) for d in range(LSTM_SEQ_LEN)]

    ref_traj_sample = REFERENCE_PLOT_RESULTS.get(key)
    hybrid_traj_sample = test_results.get(key) or train_results.get(key)
    # Every replicate, not just this Plotnumber. The model reads Year, Location,
    # Cultivar, Nitrogen and Irrigation — never Plotnumber — so the 1-3
    # replicate plots in this treatment cell all receive a byte-identical
    # predicted curve. Drawing one replicate's samples makes every model look
    # like it is missing the data, when it is really aimed at the middle of a
    # spread you could not see.
    obs_for_lstm = obs_df[
        (obs_df["Year"] == sample_plot.Year)
        & (obs_df["Location"] == sample_plot.Location)
        & (obs_df["Cultivar"] == sample_plot.Cultivar)
        & (obs_df["Nitrogen"] == sample_plot.Nitrogen)
        & (obs_df["Irrigation"] == sample_plot.Irrigation)
    ]
    n_reps = obs_for_lstm["Plotnumber"].nunique()

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    for ax, (var, obs_col, title, lstm_idx) in zip(axes.ravel(), ORGAN_LIST_LSTM):
        if ref_traj_sample is not None and var in ref_traj_sample:
            ax.plot(ref_traj_sample["day"], ref_traj_sample[var].detach().cpu().numpy(),
                    label="WOFOST72_PP", linewidth=2, color="tab:blue")
        if hybrid_traj_sample is not None and var in hybrid_traj_sample:
            ax.plot(hybrid_traj_sample["day"], hybrid_traj_sample[var].detach().cpu().numpy(),
                    label="Hybrid (PP + NN stress)", linewidth=2, linestyle="--", color="tab:orange")
        ax.plot(lstm_dates, lstm_traj[:, lstm_idx],
                label="Pure LSTM (no physics)", linewidth=2, linestyle=":", color="tab:purple")
        if obs_col in obs_for_lstm.columns:
            finite_obs = obs_for_lstm[obs_for_lstm[obs_col].notna()]
            if len(finite_obs):
                ax.scatter(finite_obs["Date"], finite_obs[obs_col],
                           s=38, color="black", zorder=5, alpha=0.75,
                           label=f"Observed ({n_reps} replicate{'s' if n_reps != 1 else ''})")
        ax.set_title(title); ax.set_ylabel("kg/ha (LAI: m²/m²)")
        ax.grid(alpha=0.3); ax.tick_params(axis="x", rotation=30)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False, fontsize=10)
    plot_label = (
        f"[{split_name}] WOFOST + hybrid + LSTM — "
        f"{sample_plot.Cultivar}@{sample_plot.Location} "
        f"{sample_plot.Nitrogen}{sample_plot.Irrigation} {sample_plot.Year}"
    )
    if n_reps > 1:
        plot_label += (
            f"\n{n_reps} replicate plots, one shared prediction — "
            "the model never reads Plotnumber"
        )
    fig.suptitle(plot_label, y=1.03, fontsize=13)
    plt.tight_layout(rect=[0, 0.06, 1, 1]); plt.show()


def _find_plot(plots, cultivar, location, irrigation, nitrogen):
    """First plot in `plots` matching a treatment cell (replicates are
    interchangeable — they share one prediction)."""
    for p in plots:
        if (p.Cultivar == cultivar and p.Location == location
                and p.Irrigation == irrigation and p.Nitrogen == nitrogen):
            return p
    return None


# One train example for reference, then the N gradient on the TEST year.
# We hold cultivar, site and water fixed and step through N0 -> N1 -> N2.
# N2 can sit at or above potential production (RFTRA can only reduce from PP),
# so that panel is a ceiling check, not the treatment we expect to fit worst.
N_ROW = dict(cultivar="C3", location="L", irrigation="W2")

if train_plots:
    plot_three_models(train_plots[0], "TRAIN")

for _N in ["N0", "N1", "N2"]:
    _p = _find_plot(test_plots, nitrogen=_N, **N_ROW)
    if _p is not None:
        plot_three_models(_p, "TEST")
    else:
        print(f"(no test plot for {N_ROW} {_N})")


Does tuber-yield accuracy vary with nitrogen treatment?

The single-plot figures above show one treatment cell at a time. This rolls the
whole test set up by nitrogen level, reported on tuber yield (`TWSO`).

`RFTRA` can only pull growth down from potential production, so any plot that
out-yields PP is unreachable by construction. That is rare at N0 and common at
N2. That does **not** mean the hybrid's TWSO error is largest at N2. On the
year split, normalised TWSO RMSE is largest at N0 (a real deficit relative to
PP) and smallest at N2 (PP is already close to the observations). The bars
show that pattern; they also show whether the hybrid still beats WOFOST72_PP.


In [ ]:
#@title Plot: three models across N treatments (test set) { display-mode: "form" }
#@markdown §13 — per-nitrogen tuber-yield RMSE, all three models.

N_LEVELS = ["N0", "N1", "N2"]
N_LABELS = {"N0": "N0\n(0% N)", "N1": "N1\n(30% N)", "N2": "N2\n(130% N)"}
# Report tuber yield (TWSO) specifically. Pooling all four organs would let
# stems (TWST, ~5x the RMSE scale) dominate the bars.
SUMMARY_VAR = "TWSO"


def _var_rmse(diag):
    vals = diag.get(SUMMARY_VAR)
    return float(np.mean(vals)) if vals else np.nan


ref_by_N, hyb_by_N, lstm_by_N, counts = [], [], [], []
for N in N_LEVELS:
    plots_N = [p for p in test_plots if p.Nitrogen == N]
    counts.append(len(plots_N))
    ref_by_N.append(_var_rmse(per_plot_diag(plots_N, REFERENCE_PLOT_RESULTS)))
    hyb_by_N.append(_var_rmse(per_plot_diag(plots_N, test_results)))
    subset = {p: lstm_data_test[p] for p in plots_N if p in lstm_data_test}
    _, lstm_diag_N, _ = lstm_pooled_loss(pure_lstm, subset, PURE_VARIABLE_WEIGHTS)
    lstm_by_N.append(float(lstm_diag_N[SUMMARY_VAR]))

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(N_LEVELS)); width = 0.27
ax.bar(x - width, ref_by_N, width, label="WOFOST72_PP",
       color="lightgray", edgecolor="black")
ax.bar(x, hyb_by_N, width, label="Hybrid (PP + NN stress)",
       color="tab:orange", edgecolor="black")
ax.bar(x + width, lstm_by_N, width, label="Pure LSTM (no physics)",
       color="tab:purple", edgecolor="black")
ax.set_xticks(x)
ax.set_xticklabels([N_LABELS[N] for N in N_LEVELS])
ax.set_ylabel(f"normalised {SUMMARY_VAR} RMSE on TEST set")
ax.set_title("Tuber-yield accuracy across nitrogen treatments — test year")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

print(f"normalised {SUMMARY_VAR} RMSE by N level (test set, lower is better):")
print(f" {'N':4}{'n_plots':>8}{'WOFOST':>9}{'hybrid':>9}{'LSTM':>9}")
for N, c, r, h, l in zip(N_LEVELS, counts, ref_by_N, hyb_by_N, lstm_by_N):
    print(f" {N:4}{c:>8}{r:9.3f}{h:9.3f}{l:9.3f}")
print("\nN0 rests on 1 replicate per cell vs 3 for N1/N2 — least reliable bar.")
print("On this year split, hybrid TWSO nRMSE is largest at N0 and smallest at N2.")


**The PP ceiling, and why N2 is not the worst bar.** The boxes below are data
only: observed final tuber yield divided by potential-production TWSO.

`RFTRA` lives in `[0, 1]`, so the hybrid can only pull yield down from PP. At
N0 the crop sits well below the ceiling — a real assimilate deficit. At N2 a
large fraction of test plots sit at or above PP. Those points are unreachable
for the hybrid. They also leave little useful work for a reduction factor:
WOFOST72_PP is already close, so normalised TWSO RMSE is *smallest* at N2, not
largest.

The red line is the ceiling. Points above it cannot be fitted by any
`RFTRA ≤ 1`. That is a bound on N2, not a prediction that N2 dominates the
error bars above.


In [ ]:
#@title Plot: the PP ceiling by N treatment { display-mode: "form" }
#@markdown §13 — observed tuber yield / potential production, by N (data only).

def _obs_final_tuber(plot):
    rows = obs_df[
        (obs_df["Year"] == plot.Year)
        & (obs_df["Location"] == plot.Location)
        & (obs_df["Plotnumber"] == plot.Plotnumber)
        & obs_df["tubersDW"].notna()
    ].sort_values("Date")
    return None if rows.empty else float(rows["tubersDW"].iloc[-1])


def _pp_ceiling(plot):
    # The reference (default WOFOST72_PP) run IS potential production; its final
    # tuber mass at maturity is the ceiling the hybrid is bounded by.
    r = REFERENCE_PLOT_RESULTS.get((plot.Year, plot.Location, plot.Plotnumber))
    return None if r is None else float(r["TWSO"][-1])


ratios_by_N = {N: [] for N in N_LEVELS}
for p in test_plots:
    obs, pp = _obs_final_tuber(p), _pp_ceiling(p)
    if obs is not None and pp and pp > 0:
        ratios_by_N[p.Nitrogen].append(obs / pp)

fig, ax = plt.subplots(figsize=(9, 5))
data = [ratios_by_N[N] for N in N_LEVELS]
bp = ax.boxplot(data, positions=np.arange(len(N_LEVELS)), widths=0.5,
                patch_artist=True, showmeans=True)
for patch in bp["boxes"]:
    patch.set_facecolor("tab:orange"); patch.set_alpha(0.35)
for i, vals in enumerate(data):
    jit = ((np.arange(len(vals)) % 5) - 2) * 0.03
    ax.scatter(np.full(len(vals), i) + jit, vals, s=20, color="black",
               alpha=0.55, zorder=3)
ax.axhline(1.0, color="red", linestyle="--", linewidth=1.5,
           label="PP ceiling — hybrid cannot exceed (RFTRA ≤ 1)")
ax.set_xticks(np.arange(len(N_LEVELS)))
ax.set_xticklabels([N_LABELS[N] for N in N_LEVELS])
ax.set_ylabel("observed final tuber yield / potential production")
ax.set_title("Where the hybrid runs out of room: obs / PP by nitrogen (test year)")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

print("plots ABOVE the PP ceiling = unfittable by the hybrid by construction:")
for N in N_LEVELS:
    v = np.array(ratios_by_N[N])
    if len(v):
        print(f" {N}: {(v > 1).mean() * 100:4.0f}% above ceiling    "
              f"median obs/PP = {np.median(v):.2f}   (n={len(v)})")


**(iii) Train vs. test gap — the headline result.** This is where the
inductive bias of the engine pays off. The LSTM tends to fit the training set
*very* well (it has the capacity to memorise a few dozen plot-years) but its
test loss is often markedly higher — a sign of overfitting. The hybrid has a
much smaller train/test ratio because the physics in the loop constrains what
shapes its predictions can take in the first place.

If you re-train the LSTM with a different seed, you'll usually see a different
test-set loss; re-training the hybrid is far more reproducible. That
**stability / consistency advantage** is one of the main reasons to keep
physics in the loop on small datasets.


In [ ]:
#@title Plot: hybrid vs LSTM generalisation { display-mode: "form" }
#@markdown §13 — pooled train/test loss comparison.

hybrid_train = final_train_loss.item()
hybrid_test  = final_test_loss.item()
lstm_train_v = final_lstm_train_loss.item()
lstm_test_v  = final_lstm_test_loss.item()

print(f"Generalisation gap (test/train) on {SPLIT_STRATEGY} split:")
print(f"  Hybrid    : train={hybrid_train:.4f}  test={hybrid_test:.4f}  "
      f"ratio={hybrid_test/max(hybrid_train,1e-9):.2f}")
print(f"  Pure LSTM : train={lstm_train_v:.4f}  test={lstm_test_v:.4f}  "
      f"ratio={lstm_test_v/max(lstm_train_v,1e-9):.2f}")

fig, ax = plt.subplots(figsize=(8, 4))
models = ["Hybrid\n(PP + NN stress)", "Pure LSTM\n(no physics)"]
trains = [hybrid_train, lstm_train_v]
tests  = [hybrid_test, lstm_test_v]
xpos = np.arange(len(models))
ax.bar(xpos - 0.2, trains, 0.4, label="train loss", color="tab:blue", edgecolor="black")
ax.bar(xpos + 0.2, tests,  0.4, label="test loss",  color="tab:orange", edgecolor="black")
ax.set_xticks(xpos); ax.set_xticklabels(models)
ax.set_ylabel("pooled normalised RMSE")
ax.set_title("Train vs. test loss — physics keeps the hybrid honest")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()


> ❓ **Discuss with a neighbour**
>
> Look at the three views above. On training data, which model wins? On test
> data? And if we would do splits by cultivar which would you expect to degrade
> more? Why?


## 14. Bonus: differentiability for free

This is the bit that makes `diffwofost` *different* from a pure forward
simulator. Because every operation in the engine is implemented in PyTorch, we
can compute gradients of *any* simulated quantity with respect to *any*
parameter — using one forward pass and one backward pass.

To demonstrate, we ask: *how much does the final tuber yield (`TWSO`) change if
each of `SPAN`, `TSUM1`, or `TSUM2` were perturbed by one unit?*

In a non-differentiable simulator, you'd run hundreds of finite-difference
simulations. With autograd, you get all sensitivities at once.


In [ ]:
sample_plot = test_plots[0] if test_plots else train_plots[0]
print(f"Sample plot: {sample_plot.Cultivar}@{sample_plot.Location} "
      f"{sample_plot.Nitrogen}{sample_plot.Irrigation} {sample_plot.Year}")


def run_with_param_overrides(plot, nn, **scalar_overrides):
    fb = PlotFeatureBuilder(
        WEATHER_FEATURES[plot.Location],
        make_plot_context_tensor(plot.Cultivar, plot.Nitrogen,
                                 plot.Irrigation, plot.Location),
    )
    cfg = build_config(NNStressFactor, et_kwargs={"nn_model": nn, "feature_builder": fb})
    pp = copy.deepcopy(parameter_provider)
    for name, value in scalar_overrides.items():
        pp.set_override(name, value, check=False)
    engine = Engine(config=cfg)
    engine.setup(
        pp,
        WEATHER_DATA_PROVIDERS[plot.Location],
        YAMLAgroManagementReader(
            agro_paths[(plot.Year, CULTIVAR_TO_VARIETY[plot.Cultivar])]
        ),
    )
    engine.run_till_terminate()
    return results_to_tensors(engine.get_output())


span_t = torch.tensor(35.0, dtype=ComputeConfig.get_dtype(), requires_grad=True)
tsum1_t = torch.tensor(150.0, dtype=ComputeConfig.get_dtype(), requires_grad=True)
tsum2_t = torch.tensor(1550.0, dtype=ComputeConfig.get_dtype(), requires_grad=True)

results_grad = run_with_param_overrides(
    sample_plot, stress_nn,
    SPAN=span_t, TSUM1=tsum1_t, TSUM2=tsum2_t,
)
twso_final = results_grad["TWSO"][-1]
print(f"\nSimulated final TWSO = {twso_final.item():.1f} kg/ha")

# ONE backward pass yields gradients to ALL parameters at once
twso_final.backward()
print()
print("Gradients of final TWSO with respect to any parameter (autograd):")
print(f"  d(TWSO)/d(SPAN)  = {span_t.grad.item():+.2f} kg/ha per day")
print(f"  d(TWSO)/d(TSUM1) = {tsum1_t.grad.item():+.4f} kg/ha per (degC*day)")
print(f"  d(TWSO)/d(TSUM2) = {tsum2_t.grad.item():+.4f} kg/ha per (degC*day)")


Translated into "% change in final TWSO per typical-magnitude
perturbation", the sensitivities look like this:


In [ ]:
#@title Plot: yield parameter sensitivities { display-mode: "form" }
#@markdown §14 — % change in final TWSO per perturbation.

TYPICAL_PERTURB = {"SPAN": 5.0, "TSUM1": 50.0, "TSUM2": 100.0}
gradients = {
    "SPAN":  span_t.grad.item(),
    "TSUM1": tsum1_t.grad.item(),
    "TSUM2": tsum2_t.grad.item(),
}
twso_baseline = twso_final.item()
sensitivities_pct = {
    name: (gradients[name] * TYPICAL_PERTURB[name]) / twso_baseline * 100
    for name in gradients
}

fig, ax = plt.subplots(figsize=(10, 4))
names = list(sensitivities_pct.keys())[::-1]
values = [sensitivities_pct[n] for n in names]
colors = ["tab:blue" if v >= 0 else "tab:red" for v in values]
bars = ax.barh(names, values, color=colors, edgecolor="black", height=0.6)

for bar, n, v in zip(bars, names, values):
    label = f"{v:+.1f}%   (perturbing {n} by {TYPICAL_PERTURB[n]:g} units)"
    if v >= 0:
        ax.text(max(v, 0.05) + 0.05, bar.get_y() + bar.get_height() / 2, label,
                va="center", ha="left", fontsize=10)
    else:
        ax.text(min(v, -0.05) - 0.05, bar.get_y() + bar.get_height() / 2, label,
                va="center", ha="right", fontsize=10)

ax.axvline(0, color="black", linewidth=0.7)
xlim = max(abs(v) for v in values) * 1.6 + 0.5
ax.set_xlim(-xlim, xlim)
ax.set_xlabel("% change in final TWSO per typical-magnitude perturbation")
ax.set_title(f"Yield sensitivity (baseline TWSO = {twso_baseline:.0f} kg/ha)")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout(); plt.show()


**Why this matters.**

- **Targeted calibration.** The gradient tells you *which* parameter to tune
  first when residuals appear.
- **Sensitivity analysis at scale.** Same cost per plot, regardless of how
  many parameters you probe.
- **Optimisation through the engine.** "What sowing date maximises yield under
  these climate scenarios?" is now a smooth optimisation problem.
- **Variational data assimilation.** Gradient-based DA (e.g. 4D-Var) uses
  exactly this computational structure.

The hybrid stress NN is one application of `diffwofost`. Gradients are the
underlying capability.


## 15. Recap and what's next

We just walked through the full pipeline of a hybrid crop model:

1. **Loaded a real field-trial dataset** (168 plot-years, two sites, five distinct
   cultivars with Fontane grown at both, three N levels, two W levels).
2. **Diagnosed the gap**: default WOFOST72_PP overpredicts biomass on
   water/N-stressed plots because it has no stress module. Potential production
   has to sit *above* the observations (per-cultivar parameters) for that gap
   to be learnable at all.
3. **Replaced the stress component** with a tiny NN producing daily `RFTRA`.
4. **Trained the NN end-to-end** with gradient descent through the WOFOST
   engine, on a held-out weather year.
5. **Inspected what the NN learned**: per-plot trajectories, cultivar-and-site
   residual profiles, and N-treatment ranking of the lumped gate.
6. **Compared against two reference points**: potential-production WOFOST72_PP
   and a rate-based LSTM, to see what the engine's inductive bias contributes —
   especially the smaller train/test gap.
7. **Saw the bonus**: free gradients with respect to any parameter.

### Mental model: where does each model belong?

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| **Calibrated WOFOST** | Interpretable, transferable, mechanistic | Labour-intensive to fit; assumptions break |
| **WOFOST72_PP (this notebook's reference)** | Maintained cultivar parameters; potential production | No water or N limitation |
| **Pure LSTM** | High fit on training data | Unstable, brittle on held-out weather years |
| **Hybrid (this tutorial)** | Reliable shape from physics, NN closes the deficit | Opaque NN; cannot exceed potential production |

### Where to go from here

- **Partitioning hybrid**:
  https://github.com/WUR-AI/diffWOFOST/blob/main/docs/notebooks/hybrid_partitioning_wofost72.ipynb
  applies the same idea to a *different* WOFOST component (carbon
  partitioning), useful for understanding what hybrid replacement can and
  can't do.
- **Pure calibration**: see https://github.com/WUR-AI/diffWOFOST/blob/main/docs/notebooks/optimization_phenology.ipynb (and friends) for parameter-only
  calibration (no NN) using the same differentiable engine — that's the
  apples-to-apples WOFOST comparison this notebook deliberately does not do.

### Caveats

- The trained NN is specific to this sowing schedule and weather window.
  Predictions outside the training distribution are unreliable.
- Cultivar genetics (`SPAN`, `TSUM1`, `TSUM2`, …) come from the maintained
  WOFOST parameter files, but sowing is still a generic 20 April for every plot.
- `RFTRA` can only reduce growth from potential production. Plots that
  out-yield PP (common at N2 on the test year) are unreachable by
  construction. That ceiling does not make N2 the largest hybrid TWSO
  error: PP is already close there, so the residual is smallest at high N.

That's it — happy hybrid modelling!
